# Subetapa 4.5 — modelo V1 y persistencia controlada

**SYNTHETIC / DEMO / ACADEMIC DATASET. Sin validación institucional.**

Reconstituye el candidato elegido en 4.4 con los mismos 160 registros de desarrollo. Los 40 reservados permanecen fuera del ajuste: no se comparan candidatos ni se reabre su evaluación. Consulta las métricas guardadas como evidencia previa.

Python 3.12, CPU, entorno aislado. Genera pipeline completo en skops; verifica paridad de predicciones y márgenes antes/después de cargar. La reconstrucción no reemplaza automáticamente la publicación aprobada de GitHub. [Guía 4.5](https://github.com/C-David28/essalud-ticket-system/blob/main/ml/docs/COLAB-4.5.md).


## 1. Herramientas de preparación

Código reproducible incluido desde `ml/colab/preparation.py`. Solo usa la biblioteca estándar. Las descargas se limitan a cinco archivos permitidos; no se descarga ni ejecuta código desde el dataset. Los ZIP se leen en memoria, sin extraerlos.

No necesitas modificar esta celda.


In [ ]:
"""Stage 4.3 only: immutable inputs, safe loading, dataset and environment checks.

Standard library only. No model fitting, predictions, split creation or database access.
This source is embedded into the notebook by build-colab-notebook.mjs.
"""
from collections import Counter
from hashlib import sha256
from io import BytesIO
import json
from pathlib import Path, PurePosixPath
import re
import stat
import subprocess
import sys
from urllib.error import HTTPError, URLError
from urllib.request import Request, build_opener, HTTPRedirectHandler
import zipfile

REPOSITORY = "C-David28/essalud-ticket-system"
DATASET_COMMIT = "9d3103991911b1b06b9faca6dce779013dae3036"
DATASET_VERSION = "tickets-category-dataset-v1.0.0"
PREPARATION_VERSION = "stage-4.3-preparation-v1"
INSTALLER_VERSION = "25.1.1"
DATASET_ROOT = "ml/datasets/v1.0.0/"
SOURCE = "SYNTHETIC / DEMO / ACADEMIC DATASET"
CATEGORIES = ["SOPORTE", "REDES", "INFRAESTRUCTURA", "BIOMEDICO"]
EXPECTED_HASHES = {
    DATASET_ROOT + "incidents.jsonl": "2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69",
    DATASET_ROOT + "families.json": "0f27342915a770acd019a31b7685c8a2eea8c2de7616a38ca2ac8c1f079078b7",
    "ml/docs/LABELING-GUIDE.md": "7cb0ee07c57a6fb4a857bbde3ba8fc6f6b1e41d911d81682c62b98f316899cc9",
    DATASET_ROOT + "manifest.json": "aa698ab7e4480e2a0f51085b5a5138b2267eb12a021eb08c590d58a7d11fdb61",
    "apps/api/src/domain/ticket.ts": "8cfa93c9f1e9a8a7106c46118671d38c63cfce2041e514210430d0bf7560ce3a",
}
ALLOWED_FILES = (
    DATASET_ROOT + "incidents.jsonl",
    DATASET_ROOT + "families.json",
    DATASET_ROOT + "manifest.json",
    "ml/docs/LABELING-GUIDE.md",
    "apps/api/src/domain/ticket.ts",
)
MAX_FILE_BYTES = 1_000_000
MAX_ARCHIVE_BYTES = 3_000_000


def require(condition, code):
    if not condition:
        raise ValueError(code)


def canonical_hash(payload):
    return sha256(payload.decode("utf-8").replace("\r\n", "\n").encode("utf-8")).hexdigest()


def parse_json(payload, code):
    try:
        return json.loads(payload.decode("utf-8"))
    except (ValueError, UnicodeError):
        raise ValueError(code) from None


class NoRedirect(HTTPRedirectHandler):
    # Never forward a private Authorization header to an unexpected host.
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


def load_github(token=None, opener=None):
    """Read only the allowlisted files at a fixed SHA; token stays in memory/header."""
    request_opener = opener or build_opener(NoRedirect())
    payloads = {}
    for name in ALLOWED_FILES:
        url = f"https://api.github.com/repos/{REPOSITORY}/contents/{name}?ref={DATASET_COMMIT}"
        headers = {"Accept": "application/vnd.github.raw+json",
                   "X-GitHub-Api-Version": "2022-11-28", "User-Agent": "essalud-ml-stage-4.3"}
        if token:
            headers["Authorization"] = "Bearer " + token
        try:
            with request_opener.open(Request(url, headers=headers), timeout=30) as response:
                payload = response.read(MAX_FILE_BYTES + 1)
        except HTTPError as error:
            raise RuntimeError(f"GITHUB_READ_FAILED_HTTP_{error.code}: verify commit, access or use ZIP upload") from None
        except (URLError, OSError):
            raise RuntimeError("GITHUB_READ_FAILED: network unavailable; use verified ZIP upload") from None
        require(len(payload) <= MAX_FILE_BYTES, "GITHUB_FILE_TOO_LARGE")
        payloads[name] = payload
    return payloads


def load_archive(payload):
    """Read an allowlisted git archive in memory. Never extract it to the filesystem."""
    require(isinstance(payload, bytes) and len(payload) <= MAX_ARCHIVE_BYTES, "ARCHIVE_TOO_LARGE")
    result, seen, total = {}, set(), 0
    try:
        with zipfile.ZipFile(BytesIO(payload)) as archive:
            require(len(archive.infolist()) <= 25, "ARCHIVE_TOO_MANY_ENTRIES")
            for entry in archive.infolist():
                name = entry.filename
                original = entry.orig_filename
                parts = PurePosixPath(name).parts
                require(original == name and name and "\\" not in original and not name.startswith("/") and
                        ":" not in name and ".." not in parts and name not in seen, "UNSAFE_ARCHIVE_PATH")
                seen.add(name)
                require(not stat.S_ISLNK(entry.external_attr >> 16), "ARCHIVE_SYMLINK")
                if entry.is_dir():
                    require(any(item.startswith(name) for item in ALLOWED_FILES), "UNEXPECTED_ARCHIVE_DIRECTORY")
                    continue
                require(name in ALLOWED_FILES, "UNEXPECTED_ARCHIVE_FILE")
                total += entry.file_size
                require(entry.file_size <= MAX_FILE_BYTES and total <= MAX_ARCHIVE_BYTES, "ARCHIVE_EXPANDED_TOO_LARGE")
                result[name] = archive.read(entry)
    except (zipfile.BadZipFile, RuntimeError):
        raise ValueError("INVALID_OR_ENCRYPTED_ARCHIVE") from None
    require(set(result) == set(ALLOWED_FILES), "ARCHIVE_MISSING_FILE")
    return result


def load_local_snapshot(repository_root):
    """Offline smoke tests only; same allowlist and hashes as GitHub/ZIP."""
    root = Path(repository_root)
    return {name: (root / name).read_bytes() for name in ALLOWED_FILES}


def verify_dataset(payloads):
    require(set(payloads) == set(ALLOWED_FILES), "INPUT_FILE_SET_MISMATCH")
    require(all(isinstance(value, bytes) and len(value) <= MAX_FILE_BYTES for value in payloads.values()), "INVALID_PAYLOAD")
    for name, digest in EXPECTED_HASHES.items():
        require(canonical_hash(payloads[name]) == digest, "IMMUTABLE_DATASET_HASH_MISMATCH")
    manifest = parse_json(payloads[DATASET_ROOT + "manifest.json"], "INVALID_MANIFEST_JSON")
    require(isinstance(manifest, dict), "INVALID_MANIFEST")
    require(manifest.get("dataset_version") == DATASET_VERSION and manifest.get("schema_version") == 1 and
            manifest.get("source") == SOURCE and manifest.get("hash_algorithm") == "SHA-256-UTF8-LF", "RELEASE_MISMATCH")
    require(manifest.get("categories") == CATEGORIES and manifest.get("distribution") == dict.fromkeys(CATEGORIES, 50), "CATEGORY_MANIFEST_MISMATCH")
    require(manifest.get("record_count") == 200 and manifest.get("family_count") == 40 and
            manifest.get("leakage_group_count") == 37 and manifest.get("records_per_family") == 5, "COUNT_MANIFEST_MISMATCH")
    expected_manifest_hashes = {"incidents.jsonl": EXPECTED_HASHES[DATASET_ROOT + "incidents.jsonl"],
                               "families.json": EXPECTED_HASHES[DATASET_ROOT + "families.json"],
                               "../../docs/LABELING-GUIDE.md": EXPECTED_HASHES["ml/docs/LABELING-GUIDE.md"]}
    require(manifest.get("files") == expected_manifest_hashes and manifest.get("training_status") == "NOT_STARTED" and
            manifest.get("splits_status") == "NOT_CREATED_STAGE_4_3", "FROZEN_MANIFEST_MISMATCH")
    match = re.search(r"export const CATEGORIES\s*=\s*\[([^\]]+)\]", payloads["apps/api/src/domain/ticket.ts"].decode("utf-8"))
    require(match is not None and re.findall(r"['\"]([^'\"]+)['\"]", match.group(1)) == CATEGORIES, "DOMAIN_CATEGORY_MISMATCH")
    catalog = parse_json(payloads[DATASET_ROOT + "families.json"], "INVALID_FAMILIES_JSON")
    require(isinstance(catalog, dict) and catalog.get("schema_version") == 1 and
            isinstance(catalog.get("families"), list), "INVALID_FAMILY_CATALOG")
    families = {row["scenario_family_id"]: row for row in catalog["families"]}
    require(len(families) == len(catalog["families"]) == 40, "INVALID_FAMILY_COUNT")
    lines = payloads[DATASET_ROOT + "incidents.jsonl"].decode("utf-8").splitlines()
    records = [parse_json(line.encode("utf-8"), "INVALID_RECORD_JSON") for line in lines]
    require(len(records) == 200, "INVALID_RECORD_COUNT")
    fields = {"record_id", "titulo", "descripcion", "categoria", "scenario_family_id", "leakage_group_id",
              "is_synthetic", "source", "dataset_version", "label_status", "label_rationale",
              "writing_style", "length_band", "is_boundary_case"}
    ids, inputs = set(), set()
    for row in records:
        require(isinstance(row, dict) and set(row) == fields, "INVALID_RECORD_FIELDS")
        require(isinstance(row["record_id"], str) and re.fullmatch(r"ML-\d{4}", row["record_id"]) and
                row["record_id"] not in ids, "INVALID_RECORD_ID")
        ids.add(row["record_id"])
        require(row["is_synthetic"] is True and row["source"] == SOURCE and row["dataset_version"] == DATASET_VERSION and
                row["label_status"] == "APPROVED", "INVALID_RECORD_PROVENANCE")
        require(row["categoria"] in CATEGORIES, "INVALID_CATEGORY")
        family = families.get(row["scenario_family_id"])
        require(family is not None and row["categoria"] == family["categoria"] and
                row["leakage_group_id"] == family["leakage_group_id"], "FAMILY_GROUP_MISMATCH")
        require(isinstance(row["titulo"], str) and 5 <= len(row["titulo"]) <= 200 and
                isinstance(row["descripcion"], str) and 10 <= len(row["descripcion"]) <= 5000, "INVALID_TEXT")
        text = (row["titulo"].strip(), row["descripcion"].strip())
        require(text not in inputs, "DUPLICATE_TEXT")
        inputs.add(text)
    distribution = dict(Counter(row["categoria"] for row in records))
    require(distribution == dict.fromkeys(CATEGORIES, 50), "UNBALANCED_DATASET")
    require(all(count == 5 for count in Counter(row["scenario_family_id"] for row in records).values()), "INVALID_FAMILY_SIZE")
    groups = len({row["leakage_group_id"] for row in records})
    require(groups == 37, "INVALID_GROUP_COUNT")
    report = {"dataset_version": DATASET_VERSION, "dataset_commit": DATASET_COMMIT,
              "dataset_sha256": EXPECTED_HASHES[DATASET_ROOT + "incidents.jsonl"],
              "records": len(records), "distribution": distribution, "families": len(families),
              "leakage_groups": groups, "is_synthetic": True, "training_started": False,
              "splits_created": False, "status": "DATASET_V1_VERIFIED"}
    return records, report


def parse_requirements(text):
    pins = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = re.fullmatch(r"([a-zA-Z0-9-]+)==([0-9]+(?:\.[0-9]+)+(?:\.post[0-9]+)?)", line)
        require(match is not None and match.group(1) not in pins, "INVALID_DEPENDENCY_PIN")
        pins[match.group(1)] = match.group(2)
    require(len(pins) == 18, "INCOMPLETE_DEPENDENCY_PINS")
    return pins


def check_python():
    require((3, 11) <= sys.version_info[:2] < (3, 14), "PYTHON_UNSUPPORTED: use Python 3.11, 3.12 or 3.13")


def prepare_environment(workspace, requirements_text, install=True):
    """Use a project-local venv so Colab's system packages remain untouched."""
    check_python()
    parse_requirements(requirements_text)
    workspace = Path(workspace).resolve()
    workspace.mkdir(parents=True, exist_ok=True)
    environment = workspace / ".venv"
    python = environment / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
    if not python.exists():
        # Colab images may lack ensurepip; host pip manages this empty venv directly.
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(environment)], check=True, timeout=120)
    requirements_file = workspace / "requirements-stage-4.3.txt"
    requirements_file.write_text(requirements_text, encoding="utf-8")
    if install:
        subprocess.run([sys.executable, "-m", "pip", "--isolated", "--python", str(python), "install", "--disable-pip-version-check",
                        "--only-binary=:all:", "--index-url", "https://pypi.org/simple",
                        f"pip=={INSTALLER_VERSION}", "-r", str(requirements_file)], check=True, timeout=600)
    return python


def verify_environment(python, requirements_text):
    pins = parse_requirements(requirements_text)
    code = """import importlib.metadata as m, json, sys
import numpy, scipy, pandas, sklearn, matplotlib
print(json.dumps({'python':sys.version.split()[0], 'pip_tool_version':m.version('pip'), 'packages':{name:m.version(name) for name in json.loads(sys.argv[1])}}))
"""
    result = subprocess.run([str(python), "-c", code, json.dumps(list(pins))],
                            check=True, capture_output=True, text=True, timeout=120)
    report = json.loads(result.stdout)
    require(report["packages"] == pins, "INSTALLED_DEPENDENCY_VERSION_MISMATCH")
    require(report["pip_tool_version"] == INSTALLER_VERSION, "INSTALLER_VERSION_MISMATCH")
    require((3, 11) <= tuple(map(int, report["python"].split(".")[:2])) < (3, 14), "ENVIRONMENT_PYTHON_UNSUPPORTED")
    subprocess.run([str(python), "-m", "pip", "check"], check=True, timeout=60)
    return report


def readiness_report(dataset_report, environment_report, requirements_text, source_mode, preparation_sha256):
    require(re.fullmatch(r"[0-9a-f]{64}", preparation_sha256) is not None, "INVALID_PREPARATION_HASH")
    return {"stage": "4.3", "status": "READY_FOR_STAGE_4_4", "source_mode": source_mode,
            "repository": REPOSITORY, "dataset": dataset_report, "environment": environment_report,
            "preparation_version": PREPARATION_VERSION, "preparation_source_sha256_utf8_lf": preparation_sha256,
            "requirements_sha256_utf8_lf": canonical_hash(requirements_text.encode("utf-8")),
            "training_started": False, "model_selected": False, "operational_database_access": False}


## 2. Configuración

Para el repositorio público actual deja **github_public**. Alternativas:

- **upload**: carga el ZIP exportado con `npm run ml:colab:export`; recomendado si el repositorio es privado, sin token.
- **github_secret**: opcional; solo para lectura privada automática. Colab Secrets debe contener `GITHUB_READ_TOKEN` con acceso autorizado a este notebook. Nunca escribas el valor aquí.
- **local_snapshot**: reservado a pruebas locales/CI; no usar en Colab.

Las dependencias se instalan dentro de `WORKSPACE/.venv`. El Python del experimento queda en `experiment_python`: evita usar los paquetes globales de Colab para el experimento. No se necesita GPU ni reiniciar el kernel por cambios de NumPy global.


In [ ]:
SOURCE_MODE = "github_public"
LOCAL_REPOSITORY = None  # Solo pruebas locales; no configurar en Colab.
WORKSPACE = Path("/content/essalud-ml-stage-4.5")
INSTALL_DEPENDENCIES = True
PREPARATION_SOURCE_SHA256 = "e2ac1f6e1f2eb0da013e253869899ab4800c0e007c3a959d4c459a9dcaf06c98"
REQUIREMENTS_TEXT = "# Scientific environment for stages 4.3-4.4; Python 3.11-3.13; CPU only.\n# All runtime dependencies are pinned; pip itself is only an installation tool.\nnumpy==2.2.6\nscipy==1.15.3\npandas==2.2.3\nscikit-learn==1.6.1\nmatplotlib==3.10.3\njoblib==1.5.1\nthreadpoolctl==3.6.0\ncontourpy==1.3.2\ncycler==0.12.1\nfonttools==4.58.0\nkiwisolver==1.4.8\npackaging==25.0\npillow==11.2.1\npyparsing==3.2.3\npython-dateutil==2.9.0.post0\npytz==2025.2\nsix==1.17.0\ntzdata==2025.2\n"
print("Dataset:", DATASET_VERSION)
print("Commit de datos:", DATASET_COMMIT)
print("Origen:", SOURCE_MODE)


## 4. Cargar los archivos versionados

**Público:** lectura sin autenticación de un commit inmutable. Si GitHub devuelve 404, comprueba que publicaste el commit de Dataset V1; no reemplaces su SHA por main.

**Privado:** abrir un notebook con tu sesión de GitHub no autentica automáticamente las descargas de la máquina de Colab. Usa el ZIP permitido o, solo si lo necesitas, un secreto de lectura.

No cargues un ZIP del directorio completo de tu proyecto, .env, backups ni datos de producción.


In [ ]:
def load_private_from_secret():
    try:
        from google.colab import userdata
        secret = userdata.get("GITHUB_READ_TOKEN")
    except Exception:
        raise RuntimeError("Autoriza GITHUB_READ_TOKEN en Colab Secrets o usa upload sin token.") from None
    try:
        return load_github(token=secret)
    finally:
        secret = None

if SOURCE_MODE == "github_public":
    payloads = load_github()
elif SOURCE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    require(len(uploaded) == 1, "Sube solamente dataset-v1-colab.zip")
    try:
        payloads = load_archive(bytes(next(iter(uploaded.values()))))
    finally:
        uploaded.clear()
elif SOURCE_MODE == "github_secret":
    payloads = load_private_from_secret()
elif SOURCE_MODE == "local_snapshot":
    require(LOCAL_REPOSITORY is not None, "LOCAL_REPOSITORY_REQUIRED")
    payloads = load_local_snapshot(LOCAL_REPOSITORY)
else:
    raise ValueError("SOURCE_MODE_INVALID")
print("Archivos permitidos cargados:", len(payloads))


## Verificar Dataset V1

Cinco archivos y hashes congelados; 200 ejemplos, 50 por categoría. No se escriben tickets operativos.


In [ ]:
records, dataset_report = verify_dataset(payloads)
print(json.dumps({k:v for k,v in dataset_report.items() if k not in {"training_started", "splits_created"}}, ensure_ascii=False, indent=2))


## Receta y evidencia de 4.4

Se incluyen fuentes, protocolo, particiones y reportes verificados por hashes del commit 1d4e5508bbfb41d219fa073436d2a9a620118730. Son los resultados locales ya registrados; no atribuir estos valores a una nueva evaluación en Colab. Las 18 dependencias científicas se conservan y se añaden tres para persistencia.


In [ ]:
EMBEDDED_FILES = {"ml/colab/preparation.py":"\"\"\"Stage 4.3 only: immutable inputs, safe loading, dataset and environment checks.\n\nStandard library only. No model fitting, predictions, split creation or database access.\nThis source is embedded into the notebook by build-colab-notebook.mjs.\n\"\"\"\nfrom collections import Counter\nfrom hashlib import sha256\nfrom io import BytesIO\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport stat\nimport subprocess\nimport sys\nfrom urllib.error import HTTPError, URLError\nfrom urllib.request import Request, build_opener, HTTPRedirectHandler\nimport zipfile\n\nREPOSITORY = \"C-David28/essalud-ticket-system\"\nDATASET_COMMIT = \"9d3103991911b1b06b9faca6dce779013dae3036\"\nDATASET_VERSION = \"tickets-category-dataset-v1.0.0\"\nPREPARATION_VERSION = \"stage-4.3-preparation-v1\"\nINSTALLER_VERSION = \"25.1.1\"\nDATASET_ROOT = \"ml/datasets/v1.0.0/\"\nSOURCE = \"SYNTHETIC / DEMO / ACADEMIC DATASET\"\nCATEGORIES = [\"SOPORTE\", \"REDES\", \"INFRAESTRUCTURA\", \"BIOMEDICO\"]\nEXPECTED_HASHES = {\n    DATASET_ROOT + \"incidents.jsonl\": \"2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69\",\n    DATASET_ROOT + \"families.json\": \"0f27342915a770acd019a31b7685c8a2eea8c2de7616a38ca2ac8c1f079078b7\",\n    \"ml/docs/LABELING-GUIDE.md\": \"7cb0ee07c57a6fb4a857bbde3ba8fc6f6b1e41d911d81682c62b98f316899cc9\",\n    DATASET_ROOT + \"manifest.json\": \"aa698ab7e4480e2a0f51085b5a5138b2267eb12a021eb08c590d58a7d11fdb61\",\n    \"apps/api/src/domain/ticket.ts\": \"8cfa93c9f1e9a8a7106c46118671d38c63cfce2041e514210430d0bf7560ce3a\",\n}\nALLOWED_FILES = (\n    DATASET_ROOT + \"incidents.jsonl\",\n    DATASET_ROOT + \"families.json\",\n    DATASET_ROOT + \"manifest.json\",\n    \"ml/docs/LABELING-GUIDE.md\",\n    \"apps/api/src/domain/ticket.ts\",\n)\nMAX_FILE_BYTES = 1_000_000\nMAX_ARCHIVE_BYTES = 3_000_000\n\n\ndef require(condition, code):\n    if not condition:\n        raise ValueError(code)\n\n\ndef canonical_hash(payload):\n    return sha256(payload.decode(\"utf-8\").replace(\"\\r\\n\", \"\\n\").encode(\"utf-8\")).hexdigest()\n\n\ndef parse_json(payload, code):\n    try:\n        return json.loads(payload.decode(\"utf-8\"))\n    except (ValueError, UnicodeError):\n        raise ValueError(code) from None\n\n\nclass NoRedirect(HTTPRedirectHandler):\n    # Never forward a private Authorization header to an unexpected host.\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        return None\n\n\ndef load_github(token=None, opener=None):\n    \"\"\"Read only the allowlisted files at a fixed SHA; token stays in memory/header.\"\"\"\n    request_opener = opener or build_opener(NoRedirect())\n    payloads = {}\n    for name in ALLOWED_FILES:\n        url = f\"https://api.github.com/repos/{REPOSITORY}/contents/{name}?ref={DATASET_COMMIT}\"\n        headers = {\"Accept\": \"application/vnd.github.raw+json\",\n                   \"X-GitHub-Api-Version\": \"2022-11-28\", \"User-Agent\": \"essalud-ml-stage-4.3\"}\n        if token:\n            headers[\"Authorization\"] = \"Bearer \" + token\n        try:\n            with request_opener.open(Request(url, headers=headers), timeout=30) as response:\n                payload = response.read(MAX_FILE_BYTES + 1)\n        except HTTPError as error:\n            raise RuntimeError(f\"GITHUB_READ_FAILED_HTTP_{error.code}: verify commit, access or use ZIP upload\") from None\n        except (URLError, OSError):\n            raise RuntimeError(\"GITHUB_READ_FAILED: network unavailable; use verified ZIP upload\") from None\n        require(len(payload) <= MAX_FILE_BYTES, \"GITHUB_FILE_TOO_LARGE\")\n        payloads[name] = payload\n    return payloads\n\n\ndef load_archive(payload):\n    \"\"\"Read an allowlisted git archive in memory. Never extract it to the filesystem.\"\"\"\n    require(isinstance(payload, bytes) and len(payload) <= MAX_ARCHIVE_BYTES, \"ARCHIVE_TOO_LARGE\")\n    result, seen, total = {}, set(), 0\n    try:\n        with zipfile.ZipFile(BytesIO(payload)) as archive:\n            require(len(archive.infolist()) <= 25, \"ARCHIVE_TOO_MANY_ENTRIES\")\n            for entry in archive.infolist():\n                name = entry.filename\n                original = entry.orig_filename\n                parts = PurePosixPath(name).parts\n                require(original == name and name and \"\\\\\" not in original and not name.startswith(\"/\") and\n                        \":\" not in name and \"..\" not in parts and name not in seen, \"UNSAFE_ARCHIVE_PATH\")\n                seen.add(name)\n                require(not stat.S_ISLNK(entry.external_attr >> 16), \"ARCHIVE_SYMLINK\")\n                if entry.is_dir():\n                    require(any(item.startswith(name) for item in ALLOWED_FILES), \"UNEXPECTED_ARCHIVE_DIRECTORY\")\n                    continue\n                require(name in ALLOWED_FILES, \"UNEXPECTED_ARCHIVE_FILE\")\n                total += entry.file_size\n                require(entry.file_size <= MAX_FILE_BYTES and total <= MAX_ARCHIVE_BYTES, \"ARCHIVE_EXPANDED_TOO_LARGE\")\n                result[name] = archive.read(entry)\n    except (zipfile.BadZipFile, RuntimeError):\n        raise ValueError(\"INVALID_OR_ENCRYPTED_ARCHIVE\") from None\n    require(set(result) == set(ALLOWED_FILES), \"ARCHIVE_MISSING_FILE\")\n    return result\n\n\ndef load_local_snapshot(repository_root):\n    \"\"\"Offline smoke tests only; same allowlist and hashes as GitHub/ZIP.\"\"\"\n    root = Path(repository_root)\n    return {name: (root / name).read_bytes() for name in ALLOWED_FILES}\n\n\ndef verify_dataset(payloads):\n    require(set(payloads) == set(ALLOWED_FILES), \"INPUT_FILE_SET_MISMATCH\")\n    require(all(isinstance(value, bytes) and len(value) <= MAX_FILE_BYTES for value in payloads.values()), \"INVALID_PAYLOAD\")\n    for name, digest in EXPECTED_HASHES.items():\n        require(canonical_hash(payloads[name]) == digest, \"IMMUTABLE_DATASET_HASH_MISMATCH\")\n    manifest = parse_json(payloads[DATASET_ROOT + \"manifest.json\"], \"INVALID_MANIFEST_JSON\")\n    require(isinstance(manifest, dict), \"INVALID_MANIFEST\")\n    require(manifest.get(\"dataset_version\") == DATASET_VERSION and manifest.get(\"schema_version\") == 1 and\n            manifest.get(\"source\") == SOURCE and manifest.get(\"hash_algorithm\") == \"SHA-256-UTF8-LF\", \"RELEASE_MISMATCH\")\n    require(manifest.get(\"categories\") == CATEGORIES and manifest.get(\"distribution\") == dict.fromkeys(CATEGORIES, 50), \"CATEGORY_MANIFEST_MISMATCH\")\n    require(manifest.get(\"record_count\") == 200 and manifest.get(\"family_count\") == 40 and\n            manifest.get(\"leakage_group_count\") == 37 and manifest.get(\"records_per_family\") == 5, \"COUNT_MANIFEST_MISMATCH\")\n    expected_manifest_hashes = {\"incidents.jsonl\": EXPECTED_HASHES[DATASET_ROOT + \"incidents.jsonl\"],\n                               \"families.json\": EXPECTED_HASHES[DATASET_ROOT + \"families.json\"],\n                               \"../../docs/LABELING-GUIDE.md\": EXPECTED_HASHES[\"ml/docs/LABELING-GUIDE.md\"]}\n    require(manifest.get(\"files\") == expected_manifest_hashes and manifest.get(\"training_status\") == \"NOT_STARTED\" and\n            manifest.get(\"splits_status\") == \"NOT_CREATED_STAGE_4_3\", \"FROZEN_MANIFEST_MISMATCH\")\n    match = re.search(r\"export const CATEGORIES\\s*=\\s*\\[([^\\]]+)\\]\", payloads[\"apps/api/src/domain/ticket.ts\"].decode(\"utf-8\"))\n    require(match is not None and re.findall(r\"['\\\"]([^'\\\"]+)['\\\"]\", match.group(1)) == CATEGORIES, \"DOMAIN_CATEGORY_MISMATCH\")\n    catalog = parse_json(payloads[DATASET_ROOT + \"families.json\"], \"INVALID_FAMILIES_JSON\")\n    require(isinstance(catalog, dict) and catalog.get(\"schema_version\") == 1 and\n            isinstance(catalog.get(\"families\"), list), \"INVALID_FAMILY_CATALOG\")\n    families = {row[\"scenario_family_id\"]: row for row in catalog[\"families\"]}\n    require(len(families) == len(catalog[\"families\"]) == 40, \"INVALID_FAMILY_COUNT\")\n    lines = payloads[DATASET_ROOT + \"incidents.jsonl\"].decode(\"utf-8\").splitlines()\n    records = [parse_json(line.encode(\"utf-8\"), \"INVALID_RECORD_JSON\") for line in lines]\n    require(len(records) == 200, \"INVALID_RECORD_COUNT\")\n    fields = {\"record_id\", \"titulo\", \"descripcion\", \"categoria\", \"scenario_family_id\", \"leakage_group_id\",\n              \"is_synthetic\", \"source\", \"dataset_version\", \"label_status\", \"label_rationale\",\n              \"writing_style\", \"length_band\", \"is_boundary_case\"}\n    ids, inputs = set(), set()\n    for row in records:\n        require(isinstance(row, dict) and set(row) == fields, \"INVALID_RECORD_FIELDS\")\n        require(isinstance(row[\"record_id\"], str) and re.fullmatch(r\"ML-\\d{4}\", row[\"record_id\"]) and\n                row[\"record_id\"] not in ids, \"INVALID_RECORD_ID\")\n        ids.add(row[\"record_id\"])\n        require(row[\"is_synthetic\"] is True and row[\"source\"] == SOURCE and row[\"dataset_version\"] == DATASET_VERSION and\n                row[\"label_status\"] == \"APPROVED\", \"INVALID_RECORD_PROVENANCE\")\n        require(row[\"categoria\"] in CATEGORIES, \"INVALID_CATEGORY\")\n        family = families.get(row[\"scenario_family_id\"])\n        require(family is not None and row[\"categoria\"] == family[\"categoria\"] and\n                row[\"leakage_group_id\"] == family[\"leakage_group_id\"], \"FAMILY_GROUP_MISMATCH\")\n        require(isinstance(row[\"titulo\"], str) and 5 <= len(row[\"titulo\"]) <= 200 and\n                isinstance(row[\"descripcion\"], str) and 10 <= len(row[\"descripcion\"]) <= 5000, \"INVALID_TEXT\")\n        text = (row[\"titulo\"].strip(), row[\"descripcion\"].strip())\n        require(text not in inputs, \"DUPLICATE_TEXT\")\n        inputs.add(text)\n    distribution = dict(Counter(row[\"categoria\"] for row in records))\n    require(distribution == dict.fromkeys(CATEGORIES, 50), \"UNBALANCED_DATASET\")\n    require(all(count == 5 for count in Counter(row[\"scenario_family_id\"] for row in records).values()), \"INVALID_FAMILY_SIZE\")\n    groups = len({row[\"leakage_group_id\"] for row in records})\n    require(groups == 37, \"INVALID_GROUP_COUNT\")\n    report = {\"dataset_version\": DATASET_VERSION, \"dataset_commit\": DATASET_COMMIT,\n              \"dataset_sha256\": EXPECTED_HASHES[DATASET_ROOT + \"incidents.jsonl\"],\n              \"records\": len(records), \"distribution\": distribution, \"families\": len(families),\n              \"leakage_groups\": groups, \"is_synthetic\": True, \"training_started\": False,\n              \"splits_created\": False, \"status\": \"DATASET_V1_VERIFIED\"}\n    return records, report\n\n\ndef parse_requirements(text):\n    pins = {}\n    for line in text.splitlines():\n        line = line.strip()\n        if not line or line.startswith(\"#\"):\n            continue\n        match = re.fullmatch(r\"([a-zA-Z0-9-]+)==([0-9]+(?:\\.[0-9]+)+(?:\\.post[0-9]+)?)\", line)\n        require(match is not None and match.group(1) not in pins, \"INVALID_DEPENDENCY_PIN\")\n        pins[match.group(1)] = match.group(2)\n    require(len(pins) == 18, \"INCOMPLETE_DEPENDENCY_PINS\")\n    return pins\n\n\ndef check_python():\n    require((3, 11) <= sys.version_info[:2] < (3, 14), \"PYTHON_UNSUPPORTED: use Python 3.11, 3.12 or 3.13\")\n\n\ndef prepare_environment(workspace, requirements_text, install=True):\n    \"\"\"Use a project-local venv so Colab's system packages remain untouched.\"\"\"\n    check_python()\n    parse_requirements(requirements_text)\n    workspace = Path(workspace).resolve()\n    workspace.mkdir(parents=True, exist_ok=True)\n    environment = workspace / \".venv\"\n    python = environment / (\"Scripts/python.exe\" if sys.platform == \"win32\" else \"bin/python\")\n    if not python.exists():\n        # Colab images may lack ensurepip; host pip manages this empty venv directly.\n        subprocess.run([sys.executable, \"-m\", \"venv\", \"--without-pip\", str(environment)], check=True, timeout=120)\n    requirements_file = workspace / \"requirements-stage-4.3.txt\"\n    requirements_file.write_text(requirements_text, encoding=\"utf-8\")\n    if install:\n        subprocess.run([sys.executable, \"-m\", \"pip\", \"--isolated\", \"--python\", str(python), \"install\", \"--disable-pip-version-check\",\n                        \"--only-binary=:all:\", \"--index-url\", \"https://pypi.org/simple\",\n                        f\"pip=={INSTALLER_VERSION}\", \"-r\", str(requirements_file)], check=True, timeout=600)\n    return python\n\n\ndef verify_environment(python, requirements_text):\n    pins = parse_requirements(requirements_text)\n    code = \"\"\"import importlib.metadata as m, json, sys\nimport numpy, scipy, pandas, sklearn, matplotlib\nprint(json.dumps({'python':sys.version.split()[0], 'pip_tool_version':m.version('pip'), 'packages':{name:m.version(name) for name in json.loads(sys.argv[1])}}))\n\"\"\"\n    result = subprocess.run([str(python), \"-c\", code, json.dumps(list(pins))],\n                            check=True, capture_output=True, text=True, timeout=120)\n    report = json.loads(result.stdout)\n    require(report[\"packages\"] == pins, \"INSTALLED_DEPENDENCY_VERSION_MISMATCH\")\n    require(report[\"pip_tool_version\"] == INSTALLER_VERSION, \"INSTALLER_VERSION_MISMATCH\")\n    require((3, 11) <= tuple(map(int, report[\"python\"].split(\".\")[:2])) < (3, 14), \"ENVIRONMENT_PYTHON_UNSUPPORTED\")\n    subprocess.run([str(python), \"-m\", \"pip\", \"check\"], check=True, timeout=60)\n    return report\n\n\ndef readiness_report(dataset_report, environment_report, requirements_text, source_mode, preparation_sha256):\n    require(re.fullmatch(r\"[0-9a-f]{64}\", preparation_sha256) is not None, \"INVALID_PREPARATION_HASH\")\n    return {\"stage\": \"4.3\", \"status\": \"READY_FOR_STAGE_4_4\", \"source_mode\": source_mode,\n            \"repository\": REPOSITORY, \"dataset\": dataset_report, \"environment\": environment_report,\n            \"preparation_version\": PREPARATION_VERSION, \"preparation_source_sha256_utf8_lf\": preparation_sha256,\n            \"requirements_sha256_utf8_lf\": canonical_hash(requirements_text.encode(\"utf-8\")),\n            \"training_started\": False, \"model_selected\": False, \"operational_database_access\": False}\n","ml/colab/requirements.txt":"# Scientific environment for stages 4.3-4.4; Python 3.11-3.13; CPU only.\n# All runtime dependencies are pinned; pip itself is only an installation tool.\nnumpy==2.2.6\nscipy==1.15.3\npandas==2.2.3\nscikit-learn==1.6.1\nmatplotlib==3.10.3\njoblib==1.5.1\nthreadpoolctl==3.6.0\ncontourpy==1.3.2\ncycler==0.12.1\nfonttools==4.58.0\nkiwisolver==1.4.8\npackaging==25.0\npillow==11.2.1\npyparsing==3.2.3\npython-dateutil==2.9.0.post0\npytz==2025.2\nsix==1.17.0\ntzdata==2025.2\n","ml/experimentation/experiment.py":"\"\"\"Stage 4.4: frozen grouped evaluation. No database, HTTP service or model export.\n\nOnly title and description enter estimators. Final test is a separate explicit phase.\n\"\"\"\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom hashlib import sha256\nimport argparse\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport time\n\n# Colab writes preparation.py beside this file; the repository reuses the 4.3 helper.\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"colab\"))\nimport preparation as prep\n\nVERSION = \"ticket-category-experiment-v1.0.0\"\nLABELS = prep.CATEGORIES\n\n\ndef digest(value):\n    return sha256(json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\",\", \":\")).encode()).hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\"utf-8\"))\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + \"\\n\", encoding=\"utf-8\")\n\n\ndef text_input(row):\n    # Never pass category, rationale, site, ID, family or group to the vectorizers.\n    return row[\"titulo\"].strip() + \"\\n\" + row[\"descripcion\"].strip()\n\n\ndef group_holdout(records, seed=42):\n    \"\"\"Exact 10/class selection using labels/groups only, before any fit or text scoring.\"\"\"\n    counts = {}\n    for row in records:\n        counts.setdefault(row[\"leakage_group_id\"], Counter())[row[\"categoria\"]] += 1\n    ordered = sorted(counts, key=lambda group: sha256(f\"{seed}|{group}\".encode()).hexdigest())\n    states = {(0, 0, 0, 0): ()}\n    target = (10, 10, 10, 10)\n    for group in ordered:\n        vector = tuple(counts[group][label] for label in LABELS)\n        for current, chosen in list(states.items()):\n            new = tuple(a + b for a, b in zip(current, vector))\n            if all(a <= b for a, b in zip(new, target)):\n                states.setdefault(new, chosen + (group,))\n    prep.require(target in states, \"NO_BALANCED_GROUP_HOLDOUT\")\n    return set(states[target])\n\n\ndef validate_protocol(protocol):\n    prep.require(protocol[\"experiment_version\"] == VERSION and protocol[\"dataset_sha256\"] == prep.EXPECTED_HASHES[prep.DATASET_ROOT + \"incidents.jsonl\"], \"PROTOCOL_DATASET_MISMATCH\")\n    prep.require(protocol[\"dataset_commit\"] == prep.DATASET_COMMIT and protocol[\"input_columns\"] == [\"titulo\", \"descripcion\"], \"PROTOCOL_INPUT_MISMATCH\")\n    prep.require(protocol[\"persist_model\"] is False and protocol[\"institutional_validation\"] is False, \"PROTOCOL_SCOPE_MISMATCH\")\n    prep.require(protocol[\"validation\"][\"folds\"] == 4 and protocol[\"validation\"][\"group_column\"] == \"leakage_group_id\", \"PROTOCOL_GROUP_MISMATCH\")\n\n\ndef freeze_splits(records, protocol):\n    import numpy as np\n    from sklearn.model_selection import StratifiedGroupKFold\n    validate_protocol(protocol)\n    held_groups = group_holdout(records, protocol[\"seed\"])\n    development = [r for r in records if r[\"leakage_group_id\"] not in held_groups]\n    test = [r for r in records if r[\"leakage_group_id\"] in held_groups]\n    y = np.array([r[\"categoria\"] for r in development])\n    groups = np.array([r[\"leakage_group_id\"] for r in development])\n    cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=protocol[\"seed\"])\n    folds = [{\"fold\": number + 1,\n              \"training_ids\": [development[i][\"record_id\"] for i in train],\n              \"validation_ids\": [development[i][\"record_id\"] for i in valid]}\n             for number, (train, valid) in enumerate(cv.split(np.zeros(len(y)), y, groups))]\n    result = {\"schema_version\": 1, \"experiment_version\": VERSION, \"dataset_sha256\": protocol[\"dataset_sha256\"],\n              \"protocol_sha256_canonical_json\": digest(protocol), \"seed\": protocol[\"seed\"],\n              \"method\": protocol[\"holdout\"][\"method\"],\n              \"development_ids\": [r[\"record_id\"] for r in development],\n              \"final_test_ids\": [r[\"record_id\"] for r in test], \"folds\": folds}\n    validate_splits(records, protocol, result)\n    return result\n\n\ndef validate_splits(records, protocol, splits):\n    validate_protocol(protocol)\n    prep.require(splits[\"protocol_sha256_canonical_json\"] == digest(protocol) and splits[\"dataset_sha256\"] == protocol[\"dataset_sha256\"], \"SPLIT_PROTOCOL_MISMATCH\")\n    by_id = {r[\"record_id\"]: r for r in records}\n    all_ids = set(by_id)\n    dev, test = set(splits[\"development_ids\"]), set(splits[\"final_test_ids\"])\n    prep.require(len(dev) == len(splits[\"development_ids\"]) == 160 and len(test) == len(splits[\"final_test_ids\"]) == 40, \"SPLIT_SIZE_MISMATCH\")\n    prep.require(not dev & test and dev | test == all_ids, \"SPLIT_COVERAGE_MISMATCH\")\n    groups = lambda ids: {by_id[i][\"leakage_group_id\"] for i in ids}\n    prep.require(not groups(dev) & groups(test), \"HOLDOUT_GROUP_LEAKAGE\")\n    prep.require(Counter(by_id[i][\"categoria\"] for i in test) == dict.fromkeys(LABELS, 10), \"HOLDOUT_CLASS_MISMATCH\")\n    validation_ids = []\n    prep.require(len(splits[\"folds\"]) == 4, \"CV_FOLD_COUNT_MISMATCH\")\n    for fold in splits[\"folds\"]:\n        train, valid = set(fold[\"training_ids\"]), set(fold[\"validation_ids\"])\n        prep.require(len(train) == len(fold[\"training_ids\"]) and len(valid) == len(fold[\"validation_ids\"]), \"CV_DUPLICATE_IDS\")\n        prep.require(not train & valid and train | valid == dev, \"CV_COVERAGE_MISMATCH\")\n        prep.require(not groups(train) & groups(valid), \"CV_GROUP_LEAKAGE\")\n        prep.require({by_id[i][\"categoria\"] for i in train} == set(LABELS) and {by_id[i][\"categoria\"] for i in valid} == set(LABELS), \"CV_CLASS_MISSING\")\n        validation_ids += fold[\"validation_ids\"]\n    prep.require(Counter(validation_ids) == Counter({i: 1 for i in dev}), \"OOF_COVERAGE_MISMATCH\")\n    return {\"development\": {\"records\": len(dev), \"groups\": len(groups(dev)), \"distribution\": dict(Counter(by_id[i][\"categoria\"] for i in dev))},\n            \"final_test\": {\"records\": len(test), \"groups\": len(groups(test)), \"distribution\": dict(Counter(by_id[i][\"categoria\"] for i in test))},\n            \"folds\": [{\"fold\": f[\"fold\"], \"training\": len(f[\"training_ids\"]), \"validation\": len(f[\"validation_ids\"]),\n                       \"validation_distribution\": dict(Counter(by_id[i][\"categoria\"] for i in f[\"validation_ids\"]))} for f in splits[\"folds\"]]}\n\n\ndef make_estimator(spec, protocol):\n    from sklearn.dummy import DummyClassifier\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.pipeline import FeatureUnion, Pipeline\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.svm import LinearSVC\n    if spec[\"algorithm\"] == \"DummyClassifier\":\n        return DummyClassifier(strategy=spec[\"strategy\"], random_state=protocol[\"seed\"])\n    vector = protocol[\"vectorizer\"]\n    shared = {key: vector[key] for key in [\"lowercase\", \"strip_accents\", \"sublinear_tf\", \"min_df\"]}\n    features = FeatureUnion([\n        (\"word\", TfidfVectorizer(analyzer=\"word\", ngram_range=tuple(vector[\"word_ngram_range\"]), stop_words=None, **shared)),\n        (\"char\", TfidfVectorizer(analyzer=\"char_wb\", ngram_range=tuple(vector[\"char_ngram_range\"]), **shared))],\n        transformer_weights={\"word\": vector[\"view_weights\"][0], \"char\": vector[\"view_weights\"][1]})\n    if spec[\"algorithm\"] == \"LogisticRegression\":\n        classifier = LogisticRegression(C=spec[\"C\"], solver=spec[\"solver\"], max_iter=spec[\"max_iter\"], random_state=protocol[\"seed\"])\n    else:\n        prep.require(spec[\"algorithm\"] == \"LinearSVC\", \"UNSUPPORTED_CANDIDATE\")\n        classifier = LinearSVC(C=spec[\"C\"], dual=spec[\"dual\"], max_iter=spec[\"max_iter\"], random_state=protocol[\"seed\"])\n    return Pipeline([(\"features\", features), (\"classifier\", classifier)])\n\n\ndef metrics(y, predictions):\n    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\n    report = classification_report(y, predictions, labels=LABELS, output_dict=True, zero_division=0)\n    matrix = confusion_matrix(y, predictions, labels=LABELS)\n    normalized = confusion_matrix(y, predictions, labels=LABELS, normalize=\"true\")\n    return {\"accuracy\": float(accuracy_score(y, predictions)), \"macro_f1\": float(f1_score(y, predictions, labels=LABELS, average=\"macro\", zero_division=0)),\n            \"weighted_f1\": float(f1_score(y, predictions, labels=LABELS, average=\"weighted\", zero_division=0)),\n            \"classification_report\": report, \"labels\": LABELS, \"confusion_matrix\": matrix.tolist(), \"confusion_matrix_normalized\": normalized.tolist()}\n\n\ndef meets_gates(macro, per_class, baseline_macro, protocol):\n    gates = protocol[\"selection\"]\n    return macro >= gates[\"minimum_macro_f1\"] and all(per_class[label][\"recall\"] >= gates[\"minimum_recall_per_category\"] for label in LABELS) and macro - baseline_macro >= gates[\"minimum_macro_f1_gain_over_baseline\"]\n\n\ndef choose_candidate(comparison, protocol):\n    \"\"\"This function receives development metrics only, never a final-test report.\"\"\"\n    rows = comparison[\"candidates\"]\n    baseline = next(r for r in rows if r[\"algorithm\"] == \"DummyClassifier\")\n    candidates = [r for r in rows if r[\"algorithm\"] != \"DummyClassifier\"]\n    qualified = [r for r in candidates if meets_gates(r[\"mean_macro_f1\"], r[\"oof_metrics\"][\"classification_report\"], baseline[\"mean_macro_f1\"], protocol)]\n    pool = qualified or candidates\n    best = max(pool, key=lambda r: (r[\"mean_macro_f1\"], r[\"id\"]))\n    nearby = [r for r in pool if best[\"mean_macro_f1\"] - r[\"mean_macro_f1\"] <= max(best[\"std_macro_f1\"], r[\"std_macro_f1\"])]\n    ranks = {name: index for index, name in enumerate(protocol[\"selection\"][\"tie_preference\"])}\n    chosen = min(nearby, key=lambda r: (ranks[r[\"algorithm\"]], r[\"spec\"][\"C\"], -r[\"mean_macro_f1\"]))\n    return {\"candidate_id\": chosen[\"id\"], \"candidate_spec\": chosen[\"spec\"], \"selection_source\": \"development_grouped_cv_only\",\n            \"cv_acceptance_passed\": bool(qualified), \"best_mean_candidate_id\": best[\"id\"],\n            \"near_tie_candidates\": [r[\"id\"] for r in nearby], \"chosen_mean_macro_f1\": chosen[\"mean_macro_f1\"],\n            \"chosen_std_macro_f1\": chosen[\"std_macro_f1\"], \"baseline_mean_macro_f1\": baseline[\"mean_macro_f1\"],\n            \"reason\": \"Eligibility gates first; CV mean/variability and predeclared algorithm/lower-C preference\" if qualified else \"No candidate passed all gates; evaluate leading development candidate, never promote on final-test performance\"}\n\n\ndef score_outputs(estimator, texts, predicted, spec):\n    import numpy as np\n    if spec[\"algorithm\"] == \"LogisticRegression\":\n        probabilities = estimator.predict_proba(texts)\n        return np.max(probabilities, axis=1).tolist(), \"uncalibrated_probability\"\n    if spec[\"algorithm\"] == \"LinearSVC\":\n        decision = estimator.decision_function(texts)\n        return np.max(decision, axis=1).tolist(), \"raw_decision_margin_not_probability\"\n    return [None] * len(predicted), \"baseline_no_confidence\"\n\n\ndef write_csv(path, rows, fieldnames=None):\n    if not rows and not fieldnames:\n        return\n    with Path(path).open(\"w\", encoding=\"utf-8\", newline=\"\") as file:\n        writer = csv.DictWriter(file, fieldnames=fieldnames or list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef compare(records, protocol, splits, output, provenance):\n    import numpy as np\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    prep.require(not (output / \"selection.json\").exists(), \"COMPARISON_EXISTS: use existing report; do not overwrite a frozen decision\")\n    # Exclude test rows before preprocessing, fitting, scoring or selection.\n    selected_ids = set(splits[\"development_ids\"])\n    development = {r[\"record_id\"]: r for r in records if r[\"record_id\"] in selected_ids}\n    rows, oof_rows = [], []\n    for spec in protocol[\"candidates\"]:\n        folds, predictions, scores, fit_times, infer_times = [], {}, {}, [], []\n        score_kind = None\n        for fold in splits[\"folds\"]:\n            train = [development[i] for i in fold[\"training_ids\"]]\n            valid = [development[i] for i in fold[\"validation_ids\"]]\n            estimator = make_estimator(spec, protocol)\n            start = time.perf_counter()\n            estimator.fit([text_input(r) for r in train], [r[\"categoria\"] for r in train])\n            fit_times.append(time.perf_counter() - start)\n            texts = [text_input(r) for r in valid]\n            start = time.perf_counter()\n            pred = estimator.predict(texts).tolist()\n            infer_times.append((time.perf_counter() - start) / len(valid))\n            values, score_kind = score_outputs(estimator, texts, pred, spec)\n            predictions.update(zip(fold[\"validation_ids\"], pred))\n            scores.update(zip(fold[\"validation_ids\"], values))\n            folds.append({\"fold\": fold[\"fold\"], **metrics([r[\"categoria\"] for r in valid], pred)})\n        ordered = splits[\"development_ids\"]\n        y = [development[i][\"categoria\"] for i in ordered]\n        pred = [predictions[i] for i in ordered]\n        row = {\"id\": spec[\"id\"], \"algorithm\": spec[\"algorithm\"], \"spec\": spec, \"folds\": folds,\n               \"mean_macro_f1\": float(np.mean([f[\"macro_f1\"] for f in folds])),\n               \"std_macro_f1\": float(np.std([f[\"macro_f1\"] for f in folds])), \"oof_metrics\": metrics(y, pred),\n               \"mean_fit_seconds\": float(np.mean(fit_times)), \"mean_inference_seconds_per_record\": float(np.mean(infer_times)),\n               \"score_kind\": score_kind}\n        if spec[\"algorithm\"] == \"LogisticRegression\":\n            curve = []\n            for threshold in protocol[\"confidence_analysis\"][\"thresholds\"]:\n                kept = [i for i in ordered if scores[i] >= threshold]\n                curve.append({\"threshold\": threshold, \"accepted\": len(kept), \"coverage\": len(kept) / len(ordered),\n                              \"accepted_accuracy\": sum(predictions[i] == development[i][\"categoria\"] for i in kept) / len(kept) if kept else None,\n                              \"abstentions_per_category\": dict(Counter(development[i][\"categoria\"] for i in ordered if i not in set(kept)))})\n            row[\"uncalibrated_probability_coverage_analysis\"] = curve\n        rows.append(row)\n        oof_rows += [{\"candidate_id\": spec[\"id\"], \"record_id\": i, \"real\": development[i][\"categoria\"], \"predicted\": predictions[i],\n                      \"correct\": development[i][\"categoria\"] == predictions[i], \"score\": scores[i], \"score_kind\": score_kind} for i in ordered]\n    comparison = {\"experiment_version\": VERSION, \"phase\": \"development_only\", \"training_executed\": True, \"splits_created\": True, \"provenance\": provenance, \"candidates\": rows}\n    decision = choose_candidate(comparison, protocol)\n    selection = {\"experiment_version\": VERSION, \"provenance\": provenance, \"decision\": decision,\n                 \"comparison_sha256_canonical_json\": digest(comparison), \"final_test_used_for_selection\": False}\n    selection[\"selection_sha256_canonical_json\"] = digest(selection)\n    write_json(output / \"comparison.json\", comparison)\n    write_csv(output / \"development-oof.csv\", oof_rows)\n    write_csv(output / \"comparison.csv\", [{\"candidate\": r[\"id\"], \"cv_macro_f1_mean\": r[\"mean_macro_f1\"], \"cv_macro_f1_std\": r[\"std_macro_f1\"],\n              \"oof_accuracy\": r[\"oof_metrics\"][\"accuracy\"], \"oof_macro_f1\": r[\"oof_metrics\"][\"macro_f1\"], \"oof_weighted_f1\": r[\"oof_metrics\"][\"weighted_f1\"],\n              \"mean_fit_seconds\": r[\"mean_fit_seconds\"], \"mean_inference_seconds_per_record\": r[\"mean_inference_seconds_per_record\"]} for r in rows])\n    plot_comparison(rows, output)\n    write_json(output / \"selection.json\", selection)\n    print(json.dumps(decision, indent=2, ensure_ascii=False))\n    return selection\n\n\ndef plot_comparison(rows, output):\n    import matplotlib\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots(figsize=(9, 4.8))\n    ax.bar([r[\"id\"] for r in rows], [r[\"mean_macro_f1\"] for r in rows], yerr=[r[\"std_macro_f1\"] for r in rows], capsize=5)\n    ax.axhline(0.70, color=\"red\", linestyle=\"--\", label=\"Objetivo académico Macro F1 = 0.70\")\n    ax.set_ylim(0, 1.05)\n    ax.set_ylabel(\"Macro F1: media y desviación entre folds\")\n    ax.set_title(\"Desarrollo: CV de 4 folds por grupos (datos sintéticos)\")\n    ax.tick_params(axis=\"x\", labelrotation=15)\n    ax.legend(fontsize=8)\n    fig.tight_layout()\n    for suffix in [\"png\", \"pdf\"]:\n        fig.savefig(Path(output) / f\"comparison.{suffix}\", dpi=180)\n    plt.close(fig)\n\n\ndef error_rows(rows, predictions, scores, kind):\n    result = []\n    for row, prediction, score in zip(rows, predictions, scores):\n        correct = row[\"categoria\"] == prediction\n        reason = \"Acierto; comprobar evidencia textual con guía\" if correct else \"Posible confusión por vocabulario compartido o familia no vista; requiere revisión humana, no es una explicación causal\"\n        if not correct and row[\"is_boundary_case\"]:\n            reason = \"Caso fronterizo: posiblemente no distingue el componente afectado; revisar manualmente según guía, sin cambiar etiqueta de prueba\"\n        result.append({\"record_id\": row[\"record_id\"], \"titulo\": row[\"titulo\"], \"descripcion\": row[\"descripcion\"],\n                       \"real\": row[\"categoria\"], \"predicted\": prediction, \"correct\": correct, \"score\": score, \"score_kind\": kind,\n                       \"scenario_family_id\": row[\"scenario_family_id\"], \"leakage_group_id\": row[\"leakage_group_id\"],\n                       \"is_boundary_case\": row[\"is_boundary_case\"], \"possible_reason_to_review\": reason})\n    return result\n\n\ndef plot_matrix(report, output, normalized=False):\n    import matplotlib\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import ConfusionMatrixDisplay\n    import numpy as np\n    key = \"confusion_matrix_normalized\" if normalized else \"confusion_matrix\"\n    fig, ax = plt.subplots(figsize=(7.5, 6))\n    ConfusionMatrixDisplay(np.array(report[key]), display_labels=LABELS).plot(ax=ax, cmap=\"Blues\", values_format=\".2f\" if normalized else \"d\", colorbar=False)\n    ax.set_title(\"Prueba final sintética: \" + (\"normalizada por fila\" if normalized else \"conteos\"))\n    ax.set_xlabel(\"Categoría predicha\")\n    ax.set_ylabel(\"Categoría real\")\n    ax.tick_params(axis=\"x\", labelrotation=15)\n    fig.tight_layout()\n    for suffix in [\"png\", \"pdf\"]:\n        fig.savefig(Path(output) / f\"{key}.{suffix}\", dpi=180)\n    plt.close(fig)\n\n\ndef evaluate_final(records, protocol, splits, output, provenance):\n    \"\"\"Read frozen decision before touching final-test texts. Refitting uses dev only.\"\"\"\n    output = Path(output)\n    selection = read_json(output / \"selection.json\")\n    comparison = read_json(output / \"comparison.json\")\n    claimed = selection.pop(\"selection_sha256_canonical_json\")\n    prep.require(digest(selection) == claimed, \"SELECTION_CHANGED\")\n    selection[\"selection_sha256_canonical_json\"] = claimed\n    prep.require(selection[\"provenance\"] == provenance and digest(comparison) == selection[\"comparison_sha256_canonical_json\"], \"EXPERIMENT_CHANGED_AFTER_SELECTION\")\n    prep.require(selection[\"decision\"] == choose_candidate(comparison, protocol) and selection[\"final_test_used_for_selection\"] is False, \"INVALID_SELECTION\")\n    prep.require(not (output / \"final-test-started.json\").exists(), \"FINAL_TEST_ALREADY_OPENED: read existing results; do not retune or overwrite\")\n    # Exclusive creation prevents two workers from opening the test in the same run.\n    with (output / \"final-test-started.json\").open(\"x\", encoding=\"utf-8\") as file:\n        json.dump({\"opened_at_utc\": datetime.now(timezone.utc).isoformat(), \"selection_sha256\": claimed}, file)\n    by_id = {r[\"record_id\"]: r for r in records}\n    development = [by_id[i] for i in splits[\"development_ids\"]]\n    final = [by_id[i] for i in splits[\"final_test_ids\"]]\n    decision = selection[\"decision\"]\n    estimator = make_estimator(decision[\"candidate_spec\"], protocol)\n    estimator.fit([text_input(r) for r in development], [r[\"categoria\"] for r in development])\n    texts = [text_input(r) for r in final]\n    predictions = estimator.predict(texts).tolist()\n    scores, kind = score_outputs(estimator, texts, predictions, decision[\"candidate_spec\"])\n    y = [r[\"categoria\"] for r in final]\n    baseline_spec = protocol[\"candidates\"][0]\n    baseline = make_estimator(baseline_spec, protocol)\n    baseline.fit([text_input(r) for r in development], [r[\"categoria\"] for r in development])\n    baseline_report = metrics(y, baseline.predict(texts).tolist())\n    report = metrics(y, predictions)\n    final_gate = meets_gates(report[\"macro_f1\"], report[\"classification_report\"], baseline_report[\"macro_f1\"], protocol)\n    accepted = decision[\"cv_acceptance_passed\"] and final_gate\n    result = {\"experiment_version\": VERSION, \"phase\": \"final_test\", \"provenance\": provenance,\n              \"training_executed\": True, \"splits_created\": True,\n              \"selection_sha256_canonical_json\": claimed, \"candidate_id\": decision[\"candidate_id\"], \"score_kind\": kind,\n              \"metrics\": report, \"baseline_metrics\": baseline_report, \"cv_acceptance_passed\": decision[\"cv_acceptance_passed\"],\n              \"final_acceptance_passed\": final_gate, \"experimental_promotion_allowed\": accepted,\n              \"institutionally_validated\": False, \"model_exported\": False,\n              \"status\": \"EXPERIMENTAL_CANDIDATE_ACCEPTED\" if accepted else \"EXPERIMENT_COMPLETE_NOT_APPROVED_FOR_PROMOTION\"}\n    examples = error_rows(final, predictions, scores, kind)\n    # Preserve measurements before rendering; only final-metrics marks full success.\n    write_json(output / \"final-evaluation.json\", {\"result\": result, \"examples\": examples})\n    write_csv(output / \"final-examples.csv\", examples)\n    write_csv(output / \"final-errors.csv\", [e for e in examples if not e[\"correct\"]], fieldnames=list(examples[0]))\n    for key in [\"confusion_matrix\", \"confusion_matrix_normalized\"]:\n        write_csv(output / f\"{key}.csv\", [{\"real\": label, **dict(zip(LABELS, values))} for label, values in zip(LABELS, report[key])])\n    plot_matrix(report, output)\n    plot_matrix(report, output, normalized=True)\n    write_report(output, selection, comparison, result, examples)\n    write_json(output / \"final-metrics.json\", result)\n    print(json.dumps(result, indent=2, ensure_ascii=False))\n    return result\n\n\ndef write_report(output, selection, comparison, final, examples):\n    decision = selection[\"decision\"]\n    lines = [\"# Experimento 4.4 — resultados reales\", \"\", \"SYNTHETIC / DEMO / ACADEMIC DATASET. Sin validación institucional.\", \"\",\n             f\"Candidato elegido solo con CV: `{decision['candidate_id']}`. Estado: **{final['status']}**.\", \"\",\n             \"160 registros de desarrollo; 40 de prueba final, 10 por categoría. Grupos indivisibles; cuatro folds congelados. Solo título y descripción entran al clasificador.\", \"\",\n             \"## Comparación de desarrollo\", \"\", \"| Candidato | Macro F1 media | Desviación | Accuracy OOF |\", \"| --- | ---: | ---: | ---: |\"]\n    lines += [f\"| {r['id']} | {r['mean_macro_f1']:.4f} | {r['std_macro_f1']:.4f} | {r['oof_metrics']['accuracy']:.4f} |\" for r in comparison[\"candidates\"]]\n    m = final[\"metrics\"]\n    lines += [\"\", f\"Selección: {decision['reason']}. Candidatos cercanos: {', '.join(decision['near_tie_candidates'])}.\", \"\", \"## Prueba final\", \"\",\n              f\"Accuracy: {m['accuracy']:.4f}; Macro F1: {m['macro_f1']:.4f}; F1 ponderado: {m['weighted_f1']:.4f}.\", \"\",\n              \"| Categoría | Precision | Recall | F1 | Soporte |\", \"| --- | ---: | ---: | ---: | ---: |\"]\n    lines += [f\"| {label} | {m['classification_report'][label]['precision']:.4f} | {m['classification_report'][label]['recall']:.4f} | {m['classification_report'][label]['f1-score']:.4f} | {int(m['classification_report'][label]['support'])} |\" for label in LABELS]\n    lines += [\"\", f\"Criterios CV: {final['cv_acceptance_passed']}; criterios prueba: {final['final_acceptance_passed']}; promoción experimental: {final['experimental_promotion_allowed']}.\", \"\",\n              \"![Comparación CV](comparison.png)\", \"\", \"![Matriz de confusión](confusion_matrix.png)\", \"\", \"![Matriz normalizada](confusion_matrix_normalized.png)\", \"\",\n              \"## Aciertos y errores\", \"\", \"El CSV final-examples incluye los 40 textos, etiquetas, predicciones y scores; final-errors contiene los fallos. Las razones automáticas son hipótesis para revisar, no explicaciones causales.\", \"\"]\n    samples = [e for e in examples if not e[\"correct\"]][:4] + [e for e in examples if e[\"correct\"]][:4]\n    for example in samples:\n        lines += [f\"- {example['record_id']}: {example['real']} → {example['predicted']}; acierto={example['correct']}. {example['possible_reason_to_review']}\"]\n    lines += [\"\", \"## Limitaciones\", \"\", \"Corpus sintético pequeño con una sola fuente asistida y sin revisión humana independiente. Los 200 registros no son 200 escenarios independientes. Con 10 casos por clase, un error cambia recall en 0.10. La desviación entre folds no es un intervalo de confianza. Probabilidades logísticas sin calibrar; márgenes SVM no son probabilidades. No se fijó un umbral operativo; la curva de cobertura solo usa OOF de desarrollo. No modificar datos, grupos o parámetros tras observar la prueba. Un fallo de criterios debe documentarse antes de considerar 4.5.\", \"\",\n              \"No se exporta modelo ni se crean servicios; 4.5 requiere autorización. GitHub/Colab deben registrar por separado dónde se ejecutó el experimento.\", \"\"]\n    Path(output, \"REPORT.md\").write_text(\"\\n\".join(lines), encoding=\"utf-8\")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--phase\", choices=[\"freeze\", \"validate\", \"compare\", \"final\"], required=True)\n    parser.add_argument(\"--input-root\", type=Path, required=True)\n    parser.add_argument(\"--protocol\", type=Path, required=True)\n    parser.add_argument(\"--splits\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    parser.add_argument(\"--requirements\", type=Path, required=True)\n    parser.add_argument(\"--source-mode\", default=\"local_snapshot\")\n    args = parser.parse_args()\n    cache = args.output.resolve() / \".matplotlib-cache\"\n    cache.mkdir(parents=True, exist_ok=True)\n    os.environ[\"MPLCONFIGDIR\"] = str(cache)\n    records, dataset = prep.verify_dataset(prep.load_local_snapshot(args.input_root))\n    protocol = read_json(args.protocol)\n    if args.phase == \"freeze\":\n        prep.require(not args.splits.exists(), \"SPLITS_ALREADY_FROZEN\")\n        args.splits.parent.mkdir(parents=True, exist_ok=True)\n        write_json(args.splits, freeze_splits(records, protocol))\n        print(\"FROZEN_SPLITS_CREATED_BEFORE_TRAINING\")\n        return\n    splits = read_json(args.splits)\n    summary = validate_splits(records, protocol, splits)\n    if args.phase == \"validate\":\n        print(json.dumps(summary, ensure_ascii=False, indent=2))\n        return\n    requirements = args.requirements.read_text(encoding=\"utf-8\")\n    env = prep.verify_environment(sys.executable, requirements)\n    # Release-level 4.3 flags describe that immutable release, not this training run.\n    dataset_provenance = {key: value for key, value in dataset.items() if key not in {\"training_started\", \"splits_created\"}}\n    provenance = {\"dataset\": dataset_provenance, \"protocol_sha256_canonical_json\": digest(protocol), \"splits_sha256_canonical_json\": digest(splits),\n                  \"experiment_source_sha256_utf8_lf\": prep.canonical_hash(Path(__file__).read_bytes()),\n                  \"preparation_source_sha256_utf8_lf\": prep.canonical_hash(Path(prep.__file__).read_bytes()),\n                  \"requirements_sha256_utf8_lf\": prep.canonical_hash(requirements.encode()), \"environment\": env, \"source_mode\": args.source_mode,\n                  \"partition_summary\": summary}\n    if args.phase == \"compare\":\n        compare(records, protocol, splits, args.output, provenance)\n    else:\n        evaluate_final(records, protocol, splits, args.output, provenance)\n\n\nif __name__ == \"__main__\":\n    try:\n        main()\n    except Exception as error:\n        print(f\"EXPERIMENT_FAILED: {type(error).__name__}: {error}\", file=sys.stderr)\n        sys.exit(1)\n","ml/experiments/experiment-v1.0.0/protocol.json":"{\n  \"schema_version\": 1,\n  \"experiment_version\": \"ticket-category-experiment-v1.0.0\",\n  \"dataset_commit\": \"9d3103991911b1b06b9faca6dce779013dae3036\",\n  \"dataset_sha256\": \"2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69\",\n  \"seed\": 42,\n  \"input_columns\": [\"titulo\", \"descripcion\"],\n  \"text_join\": \"trim(title) + LF + trim(description)\",\n  \"holdout\": {\"method\": \"sha256-seeded-group-subset-dp-v1\", \"records\": 40, \"records_per_category\": 10},\n  \"validation\": {\"method\": \"StratifiedGroupKFold\", \"folds\": 4, \"shuffle\": true, \"group_column\": \"leakage_group_id\"},\n  \"vectorizer\": {\n    \"views\": [\"word\", \"char_wb\"],\n    \"word_ngram_range\": [1, 2],\n    \"char_ngram_range\": [3, 5],\n    \"lowercase\": true,\n    \"strip_accents\": \"unicode\",\n    \"sublinear_tf\": true,\n    \"min_df\": 1,\n    \"stop_words\": null,\n    \"view_weights\": [1.0, 1.0]\n  },\n  \"candidates\": [\n    {\"id\": \"dummy-majority\", \"algorithm\": \"DummyClassifier\", \"strategy\": \"most_frequent\"},\n    {\"id\": \"logreg-c1\", \"algorithm\": \"LogisticRegression\", \"C\": 1.0, \"solver\": \"lbfgs\", \"max_iter\": 2000},\n    {\"id\": \"logreg-c4\", \"algorithm\": \"LogisticRegression\", \"C\": 4.0, \"solver\": \"lbfgs\", \"max_iter\": 2000},\n    {\"id\": \"linear-svc-c1\", \"algorithm\": \"LinearSVC\", \"C\": 1.0, \"dual\": \"auto\", \"max_iter\": 10000},\n    {\"id\": \"linear-svc-c4\", \"algorithm\": \"LinearSVC\", \"C\": 4.0, \"dual\": \"auto\", \"max_iter\": 10000}\n  ],\n  \"selection\": {\n    \"primary\": \"mean_fold_macro_f1\",\n    \"recall_gate\": \"pooled_out_of_fold_per_category\",\n    \"minimum_macro_f1\": 0.70,\n    \"minimum_recall_per_category\": 0.60,\n    \"minimum_macro_f1_gain_over_baseline\": 0.10,\n    \"tie_tolerance\": \"maximum_population_std_of_best_and_candidate_fold_macro_f1\",\n    \"tie_preference\": [\"LogisticRegression\", \"LinearSVC\"],\n    \"within_algorithm_preference\": \"lowest_C\",\n    \"no_qualified_candidate\": \"evaluate_leading_candidate_once_but_do_not_promote\"\n  },\n  \"confidence_analysis\": {\n    \"source\": \"development_out_of_fold_only\",\n    \"probabilities\": \"uncalibrated_LogisticRegression_only\",\n    \"thresholds\": [0.25, 0.4, 0.55, 0.7, 0.85],\n    \"operational_threshold\": null\n  },\n  \"final_test\": {\"selected_candidate_only\": true, \"baseline_comparison\": true, \"same_acceptance_gates\": true, \"no_retuning\": true},\n  \"institutional_validation\": false,\n  \"persist_model\": false\n}\n","ml/experiments/experiment-v1.0.0/splits.json":"{\n  \"schema_version\": 1,\n  \"experiment_version\": \"ticket-category-experiment-v1.0.0\",\n  \"dataset_sha256\": \"2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69\",\n  \"protocol_sha256_canonical_json\": \"c5eacbc7011971ad0f0ccbaf17b974866304386df5d7216869994a546fdb3959\",\n  \"seed\": 42,\n  \"method\": \"sha256-seeded-group-subset-dp-v1\",\n  \"development_ids\": [\n    \"ML-0011\",\n    \"ML-0012\",\n    \"ML-0013\",\n    \"ML-0014\",\n    \"ML-0015\",\n    \"ML-0016\",\n    \"ML-0017\",\n    \"ML-0018\",\n    \"ML-0019\",\n    \"ML-0020\",\n    \"ML-0021\",\n    \"ML-0022\",\n    \"ML-0023\",\n    \"ML-0024\",\n    \"ML-0025\",\n    \"ML-0026\",\n    \"ML-0027\",\n    \"ML-0028\",\n    \"ML-0029\",\n    \"ML-0030\",\n    \"ML-0031\",\n    \"ML-0032\",\n    \"ML-0033\",\n    \"ML-0034\",\n    \"ML-0035\",\n    \"ML-0036\",\n    \"ML-0037\",\n    \"ML-0038\",\n    \"ML-0039\",\n    \"ML-0040\",\n    \"ML-0041\",\n    \"ML-0042\",\n    \"ML-0043\",\n    \"ML-0044\",\n    \"ML-0045\",\n    \"ML-0046\",\n    \"ML-0047\",\n    \"ML-0048\",\n    \"ML-0049\",\n    \"ML-0050\",\n    \"ML-0051\",\n    \"ML-0052\",\n    \"ML-0053\",\n    \"ML-0054\",\n    \"ML-0055\",\n    \"ML-0061\",\n    \"ML-0062\",\n    \"ML-0063\",\n    \"ML-0064\",\n    \"ML-0065\",\n    \"ML-0066\",\n    \"ML-0067\",\n    \"ML-0068\",\n    \"ML-0069\",\n    \"ML-0070\",\n    \"ML-0071\",\n    \"ML-0072\",\n    \"ML-0073\",\n    \"ML-0074\",\n    \"ML-0075\",\n    \"ML-0076\",\n    \"ML-0077\",\n    \"ML-0078\",\n    \"ML-0079\",\n    \"ML-0080\",\n    \"ML-0081\",\n    \"ML-0082\",\n    \"ML-0083\",\n    \"ML-0084\",\n    \"ML-0085\",\n    \"ML-0091\",\n    \"ML-0092\",\n    \"ML-0093\",\n    \"ML-0094\",\n    \"ML-0095\",\n    \"ML-0096\",\n    \"ML-0097\",\n    \"ML-0098\",\n    \"ML-0099\",\n    \"ML-0100\",\n    \"ML-0101\",\n    \"ML-0102\",\n    \"ML-0103\",\n    \"ML-0104\",\n    \"ML-0105\",\n    \"ML-0106\",\n    \"ML-0107\",\n    \"ML-0108\",\n    \"ML-0109\",\n    \"ML-0110\",\n    \"ML-0116\",\n    \"ML-0117\",\n    \"ML-0118\",\n    \"ML-0119\",\n    \"ML-0120\",\n    \"ML-0121\",\n    \"ML-0122\",\n    \"ML-0123\",\n    \"ML-0124\",\n    \"ML-0125\",\n    \"ML-0131\",\n    \"ML-0132\",\n    \"ML-0133\",\n    \"ML-0134\",\n    \"ML-0135\",\n    \"ML-0136\",\n    \"ML-0137\",\n    \"ML-0138\",\n    \"ML-0139\",\n    \"ML-0140\",\n    \"ML-0141\",\n    \"ML-0142\",\n    \"ML-0143\",\n    \"ML-0144\",\n    \"ML-0145\",\n    \"ML-0146\",\n    \"ML-0147\",\n    \"ML-0148\",\n    \"ML-0149\",\n    \"ML-0150\",\n    \"ML-0151\",\n    \"ML-0152\",\n    \"ML-0153\",\n    \"ML-0154\",\n    \"ML-0155\",\n    \"ML-0156\",\n    \"ML-0157\",\n    \"ML-0158\",\n    \"ML-0159\",\n    \"ML-0160\",\n    \"ML-0161\",\n    \"ML-0162\",\n    \"ML-0163\",\n    \"ML-0164\",\n    \"ML-0165\",\n    \"ML-0171\",\n    \"ML-0172\",\n    \"ML-0173\",\n    \"ML-0174\",\n    \"ML-0175\",\n    \"ML-0176\",\n    \"ML-0177\",\n    \"ML-0178\",\n    \"ML-0179\",\n    \"ML-0180\",\n    \"ML-0181\",\n    \"ML-0182\",\n    \"ML-0183\",\n    \"ML-0184\",\n    \"ML-0185\",\n    \"ML-0191\",\n    \"ML-0192\",\n    \"ML-0193\",\n    \"ML-0194\",\n    \"ML-0195\",\n    \"ML-0196\",\n    \"ML-0197\",\n    \"ML-0198\",\n    \"ML-0199\",\n    \"ML-0200\"\n  ],\n  \"final_test_ids\": [\n    \"ML-0001\",\n    \"ML-0002\",\n    \"ML-0003\",\n    \"ML-0004\",\n    \"ML-0005\",\n    \"ML-0006\",\n    \"ML-0007\",\n    \"ML-0008\",\n    \"ML-0009\",\n    \"ML-0010\",\n    \"ML-0056\",\n    \"ML-0057\",\n    \"ML-0058\",\n    \"ML-0059\",\n    \"ML-0060\",\n    \"ML-0086\",\n    \"ML-0087\",\n    \"ML-0088\",\n    \"ML-0089\",\n    \"ML-0090\",\n    \"ML-0111\",\n    \"ML-0112\",\n    \"ML-0113\",\n    \"ML-0114\",\n    \"ML-0115\",\n    \"ML-0126\",\n    \"ML-0127\",\n    \"ML-0128\",\n    \"ML-0129\",\n    \"ML-0130\",\n    \"ML-0166\",\n    \"ML-0167\",\n    \"ML-0168\",\n    \"ML-0169\",\n    \"ML-0170\",\n    \"ML-0186\",\n    \"ML-0187\",\n    \"ML-0188\",\n    \"ML-0189\",\n    \"ML-0190\"\n  ],\n  \"folds\": [\n    {\n      \"fold\": 1,\n      \"training_ids\": [\n        \"ML-0011\",\n        \"ML-0012\",\n        \"ML-0013\",\n        \"ML-0014\",\n        \"ML-0015\",\n        \"ML-0016\",\n        \"ML-0017\",\n        \"ML-0018\",\n        \"ML-0019\",\n        \"ML-0020\",\n        \"ML-0021\",\n        \"ML-0022\",\n        \"ML-0023\",\n        \"ML-0024\",\n        \"ML-0025\",\n        \"ML-0031\",\n        \"ML-0032\",\n        \"ML-0033\",\n        \"ML-0034\",\n        \"ML-0035\",\n        \"ML-0041\",\n        \"ML-0042\",\n        \"ML-0043\",\n        \"ML-0044\",\n        \"ML-0045\",\n        \"ML-0046\",\n        \"ML-0047\",\n        \"ML-0048\",\n        \"ML-0049\",\n        \"ML-0050\",\n        \"ML-0051\",\n        \"ML-0052\",\n        \"ML-0053\",\n        \"ML-0054\",\n        \"ML-0055\",\n        \"ML-0061\",\n        \"ML-0062\",\n        \"ML-0063\",\n        \"ML-0064\",\n        \"ML-0065\",\n        \"ML-0066\",\n        \"ML-0067\",\n        \"ML-0068\",\n        \"ML-0069\",\n        \"ML-0070\",\n        \"ML-0071\",\n        \"ML-0072\",\n        \"ML-0073\",\n        \"ML-0074\",\n        \"ML-0075\",\n        \"ML-0076\",\n        \"ML-0077\",\n        \"ML-0078\",\n        \"ML-0079\",\n        \"ML-0080\",\n        \"ML-0091\",\n        \"ML-0092\",\n        \"ML-0093\",\n        \"ML-0094\",\n        \"ML-0095\",\n        \"ML-0101\",\n        \"ML-0102\",\n        \"ML-0103\",\n        \"ML-0104\",\n        \"ML-0105\",\n        \"ML-0106\",\n        \"ML-0107\",\n        \"ML-0108\",\n        \"ML-0109\",\n        \"ML-0110\",\n        \"ML-0116\",\n        \"ML-0117\",\n        \"ML-0118\",\n        \"ML-0119\",\n        \"ML-0120\",\n        \"ML-0121\",\n        \"ML-0122\",\n        \"ML-0123\",\n        \"ML-0124\",\n        \"ML-0125\",\n        \"ML-0141\",\n        \"ML-0142\",\n        \"ML-0143\",\n        \"ML-0144\",\n        \"ML-0145\",\n        \"ML-0146\",\n        \"ML-0147\",\n        \"ML-0148\",\n        \"ML-0149\",\n        \"ML-0150\",\n        \"ML-0151\",\n        \"ML-0152\",\n        \"ML-0153\",\n        \"ML-0154\",\n        \"ML-0155\",\n        \"ML-0161\",\n        \"ML-0162\",\n        \"ML-0163\",\n        \"ML-0164\",\n        \"ML-0165\",\n        \"ML-0171\",\n        \"ML-0172\",\n        \"ML-0173\",\n        \"ML-0174\",\n        \"ML-0175\",\n        \"ML-0176\",\n        \"ML-0177\",\n        \"ML-0178\",\n        \"ML-0179\",\n        \"ML-0180\",\n        \"ML-0191\",\n        \"ML-0192\",\n        \"ML-0193\",\n        \"ML-0194\",\n        \"ML-0195\",\n        \"ML-0196\",\n        \"ML-0197\",\n        \"ML-0198\",\n        \"ML-0199\",\n        \"ML-0200\"\n      ],\n      \"validation_ids\": [\n        \"ML-0026\",\n        \"ML-0027\",\n        \"ML-0028\",\n        \"ML-0029\",\n        \"ML-0030\",\n        \"ML-0036\",\n        \"ML-0037\",\n        \"ML-0038\",\n        \"ML-0039\",\n        \"ML-0040\",\n        \"ML-0081\",\n        \"ML-0082\",\n        \"ML-0083\",\n        \"ML-0084\",\n        \"ML-0085\",\n        \"ML-0096\",\n        \"ML-0097\",\n        \"ML-0098\",\n        \"ML-0099\",\n        \"ML-0100\",\n        \"ML-0131\",\n        \"ML-0132\",\n        \"ML-0133\",\n        \"ML-0134\",\n        \"ML-0135\",\n        \"ML-0136\",\n        \"ML-0137\",\n        \"ML-0138\",\n        \"ML-0139\",\n        \"ML-0140\",\n        \"ML-0156\",\n        \"ML-0157\",\n        \"ML-0158\",\n        \"ML-0159\",\n        \"ML-0160\",\n        \"ML-0181\",\n        \"ML-0182\",\n        \"ML-0183\",\n        \"ML-0184\",\n        \"ML-0185\"\n      ]\n    },\n    {\n      \"fold\": 2,\n      \"training_ids\": [\n        \"ML-0011\",\n        \"ML-0012\",\n        \"ML-0013\",\n        \"ML-0014\",\n        \"ML-0015\",\n        \"ML-0016\",\n        \"ML-0017\",\n        \"ML-0018\",\n        \"ML-0019\",\n        \"ML-0020\",\n        \"ML-0026\",\n        \"ML-0027\",\n        \"ML-0028\",\n        \"ML-0029\",\n        \"ML-0030\",\n        \"ML-0031\",\n        \"ML-0032\",\n        \"ML-0033\",\n        \"ML-0034\",\n        \"ML-0035\",\n        \"ML-0036\",\n        \"ML-0037\",\n        \"ML-0038\",\n        \"ML-0039\",\n        \"ML-0040\",\n        \"ML-0041\",\n        \"ML-0042\",\n        \"ML-0043\",\n        \"ML-0044\",\n        \"ML-0045\",\n        \"ML-0061\",\n        \"ML-0062\",\n        \"ML-0063\",\n        \"ML-0064\",\n        \"ML-0065\",\n        \"ML-0066\",\n        \"ML-0067\",\n        \"ML-0068\",\n        \"ML-0069\",\n        \"ML-0070\",\n        \"ML-0076\",\n        \"ML-0077\",\n        \"ML-0078\",\n        \"ML-0079\",\n        \"ML-0080\",\n        \"ML-0081\",\n        \"ML-0082\",\n        \"ML-0083\",\n        \"ML-0084\",\n        \"ML-0085\",\n        \"ML-0091\",\n        \"ML-0092\",\n        \"ML-0093\",\n        \"ML-0094\",\n        \"ML-0095\",\n        \"ML-0096\",\n        \"ML-0097\",\n        \"ML-0098\",\n        \"ML-0099\",\n        \"ML-0100\",\n        \"ML-0101\",\n        \"ML-0102\",\n        \"ML-0103\",\n        \"ML-0104\",\n        \"ML-0105\",\n        \"ML-0106\",\n        \"ML-0107\",\n        \"ML-0108\",\n        \"ML-0109\",\n        \"ML-0110\",\n        \"ML-0131\",\n        \"ML-0132\",\n        \"ML-0133\",\n        \"ML-0134\",\n        \"ML-0135\",\n        \"ML-0136\",\n        \"ML-0137\",\n        \"ML-0138\",\n        \"ML-0139\",\n        \"ML-0140\",\n        \"ML-0141\",\n        \"ML-0142\",\n        \"ML-0143\",\n        \"ML-0144\",\n        \"ML-0145\",\n        \"ML-0146\",\n        \"ML-0147\",\n        \"ML-0148\",\n        \"ML-0149\",\n        \"ML-0150\",\n        \"ML-0151\",\n        \"ML-0152\",\n        \"ML-0153\",\n        \"ML-0154\",\n        \"ML-0155\",\n        \"ML-0156\",\n        \"ML-0157\",\n        \"ML-0158\",\n        \"ML-0159\",\n        \"ML-0160\",\n        \"ML-0161\",\n        \"ML-0162\",\n        \"ML-0163\",\n        \"ML-0164\",\n        \"ML-0165\",\n        \"ML-0171\",\n        \"ML-0172\",\n        \"ML-0173\",\n        \"ML-0174\",\n        \"ML-0175\",\n        \"ML-0181\",\n        \"ML-0182\",\n        \"ML-0183\",\n        \"ML-0184\",\n        \"ML-0185\",\n        \"ML-0191\",\n        \"ML-0192\",\n        \"ML-0193\",\n        \"ML-0194\",\n        \"ML-0195\",\n        \"ML-0196\",\n        \"ML-0197\",\n        \"ML-0198\",\n        \"ML-0199\",\n        \"ML-0200\"\n      ],\n      \"validation_ids\": [\n        \"ML-0021\",\n        \"ML-0022\",\n        \"ML-0023\",\n        \"ML-0024\",\n        \"ML-0025\",\n        \"ML-0046\",\n        \"ML-0047\",\n        \"ML-0048\",\n        \"ML-0049\",\n        \"ML-0050\",\n        \"ML-0051\",\n        \"ML-0052\",\n        \"ML-0053\",\n        \"ML-0054\",\n        \"ML-0055\",\n        \"ML-0071\",\n        \"ML-0072\",\n        \"ML-0073\",\n        \"ML-0074\",\n        \"ML-0075\",\n        \"ML-0116\",\n        \"ML-0117\",\n        \"ML-0118\",\n        \"ML-0119\",\n        \"ML-0120\",\n        \"ML-0121\",\n        \"ML-0122\",\n        \"ML-0123\",\n        \"ML-0124\",\n        \"ML-0125\",\n        \"ML-0176\",\n        \"ML-0177\",\n        \"ML-0178\",\n        \"ML-0179\",\n        \"ML-0180\"\n      ]\n    },\n    {\n      \"fold\": 3,\n      \"training_ids\": [\n        \"ML-0021\",\n        \"ML-0022\",\n        \"ML-0023\",\n        \"ML-0024\",\n        \"ML-0025\",\n        \"ML-0026\",\n        \"ML-0027\",\n        \"ML-0028\",\n        \"ML-0029\",\n        \"ML-0030\",\n        \"ML-0031\",\n        \"ML-0032\",\n        \"ML-0033\",\n        \"ML-0034\",\n        \"ML-0035\",\n        \"ML-0036\",\n        \"ML-0037\",\n        \"ML-0038\",\n        \"ML-0039\",\n        \"ML-0040\",\n        \"ML-0041\",\n        \"ML-0042\",\n        \"ML-0043\",\n        \"ML-0044\",\n        \"ML-0045\",\n        \"ML-0046\",\n        \"ML-0047\",\n        \"ML-0048\",\n        \"ML-0049\",\n        \"ML-0050\",\n        \"ML-0051\",\n        \"ML-0052\",\n        \"ML-0053\",\n        \"ML-0054\",\n        \"ML-0055\",\n        \"ML-0061\",\n        \"ML-0062\",\n        \"ML-0063\",\n        \"ML-0064\",\n        \"ML-0065\",\n        \"ML-0071\",\n        \"ML-0072\",\n        \"ML-0073\",\n        \"ML-0074\",\n        \"ML-0075\",\n        \"ML-0076\",\n        \"ML-0077\",\n        \"ML-0078\",\n        \"ML-0079\",\n        \"ML-0080\",\n        \"ML-0081\",\n        \"ML-0082\",\n        \"ML-0083\",\n        \"ML-0084\",\n        \"ML-0085\",\n        \"ML-0096\",\n        \"ML-0097\",\n        \"ML-0098\",\n        \"ML-0099\",\n        \"ML-0100\",\n        \"ML-0106\",\n        \"ML-0107\",\n        \"ML-0108\",\n        \"ML-0109\",\n        \"ML-0110\",\n        \"ML-0116\",\n        \"ML-0117\",\n        \"ML-0118\",\n        \"ML-0119\",\n        \"ML-0120\",\n        \"ML-0121\",\n        \"ML-0122\",\n        \"ML-0123\",\n        \"ML-0124\",\n        \"ML-0125\",\n        \"ML-0131\",\n        \"ML-0132\",\n        \"ML-0133\",\n        \"ML-0134\",\n        \"ML-0135\",\n        \"ML-0136\",\n        \"ML-0137\",\n        \"ML-0138\",\n        \"ML-0139\",\n        \"ML-0140\",\n        \"ML-0151\",\n        \"ML-0152\",\n        \"ML-0153\",\n        \"ML-0154\",\n        \"ML-0155\",\n        \"ML-0156\",\n        \"ML-0157\",\n        \"ML-0158\",\n        \"ML-0159\",\n        \"ML-0160\",\n        \"ML-0161\",\n        \"ML-0162\",\n        \"ML-0163\",\n        \"ML-0164\",\n        \"ML-0165\",\n        \"ML-0176\",\n        \"ML-0177\",\n        \"ML-0178\",\n        \"ML-0179\",\n        \"ML-0180\",\n        \"ML-0181\",\n        \"ML-0182\",\n        \"ML-0183\",\n        \"ML-0184\",\n        \"ML-0185\",\n        \"ML-0191\",\n        \"ML-0192\",\n        \"ML-0193\",\n        \"ML-0194\",\n        \"ML-0195\",\n        \"ML-0196\",\n        \"ML-0197\",\n        \"ML-0198\",\n        \"ML-0199\",\n        \"ML-0200\"\n      ],\n      \"validation_ids\": [\n        \"ML-0011\",\n        \"ML-0012\",\n        \"ML-0013\",\n        \"ML-0014\",\n        \"ML-0015\",\n        \"ML-0016\",\n        \"ML-0017\",\n        \"ML-0018\",\n        \"ML-0019\",\n        \"ML-0020\",\n        \"ML-0066\",\n        \"ML-0067\",\n        \"ML-0068\",\n        \"ML-0069\",\n        \"ML-0070\",\n        \"ML-0091\",\n        \"ML-0092\",\n        \"ML-0093\",\n        \"ML-0094\",\n        \"ML-0095\",\n        \"ML-0101\",\n        \"ML-0102\",\n        \"ML-0103\",\n        \"ML-0104\",\n        \"ML-0105\",\n        \"ML-0141\",\n        \"ML-0142\",\n        \"ML-0143\",\n        \"ML-0144\",\n        \"ML-0145\",\n        \"ML-0146\",\n        \"ML-0147\",\n        \"ML-0148\",\n        \"ML-0149\",\n        \"ML-0150\",\n        \"ML-0171\",\n        \"ML-0172\",\n        \"ML-0173\",\n        \"ML-0174\",\n        \"ML-0175\"\n      ]\n    },\n    {\n      \"fold\": 4,\n      \"training_ids\": [\n        \"ML-0011\",\n        \"ML-0012\",\n        \"ML-0013\",\n        \"ML-0014\",\n        \"ML-0015\",\n        \"ML-0016\",\n        \"ML-0017\",\n        \"ML-0018\",\n        \"ML-0019\",\n        \"ML-0020\",\n        \"ML-0021\",\n        \"ML-0022\",\n        \"ML-0023\",\n        \"ML-0024\",\n        \"ML-0025\",\n        \"ML-0026\",\n        \"ML-0027\",\n        \"ML-0028\",\n        \"ML-0029\",\n        \"ML-0030\",\n        \"ML-0036\",\n        \"ML-0037\",\n        \"ML-0038\",\n        \"ML-0039\",\n        \"ML-0040\",\n        \"ML-0046\",\n        \"ML-0047\",\n        \"ML-0048\",\n        \"ML-0049\",\n        \"ML-0050\",\n        \"ML-0051\",\n        \"ML-0052\",\n        \"ML-0053\",\n        \"ML-0054\",\n        \"ML-0055\",\n        \"ML-0066\",\n        \"ML-0067\",\n        \"ML-0068\",\n        \"ML-0069\",\n        \"ML-0070\",\n        \"ML-0071\",\n        \"ML-0072\",\n        \"ML-0073\",\n        \"ML-0074\",\n        \"ML-0075\",\n        \"ML-0081\",\n        \"ML-0082\",\n        \"ML-0083\",\n        \"ML-0084\",\n        \"ML-0085\",\n        \"ML-0091\",\n        \"ML-0092\",\n        \"ML-0093\",\n        \"ML-0094\",\n        \"ML-0095\",\n        \"ML-0096\",\n        \"ML-0097\",\n        \"ML-0098\",\n        \"ML-0099\",\n        \"ML-0100\",\n        \"ML-0101\",\n        \"ML-0102\",\n        \"ML-0103\",\n        \"ML-0104\",\n        \"ML-0105\",\n        \"ML-0116\",\n        \"ML-0117\",\n        \"ML-0118\",\n        \"ML-0119\",\n        \"ML-0120\",\n        \"ML-0121\",\n        \"ML-0122\",\n        \"ML-0123\",\n        \"ML-0124\",\n        \"ML-0125\",\n        \"ML-0131\",\n        \"ML-0132\",\n        \"ML-0133\",\n        \"ML-0134\",\n        \"ML-0135\",\n        \"ML-0136\",\n        \"ML-0137\",\n        \"ML-0138\",\n        \"ML-0139\",\n        \"ML-0140\",\n        \"ML-0141\",\n        \"ML-0142\",\n        \"ML-0143\",\n        \"ML-0144\",\n        \"ML-0145\",\n        \"ML-0146\",\n        \"ML-0147\",\n        \"ML-0148\",\n        \"ML-0149\",\n        \"ML-0150\",\n        \"ML-0156\",\n        \"ML-0157\",\n        \"ML-0158\",\n        \"ML-0159\",\n        \"ML-0160\",\n        \"ML-0171\",\n        \"ML-0172\",\n        \"ML-0173\",\n        \"ML-0174\",\n        \"ML-0175\",\n        \"ML-0176\",\n        \"ML-0177\",\n        \"ML-0178\",\n        \"ML-0179\",\n        \"ML-0180\",\n        \"ML-0181\",\n        \"ML-0182\",\n        \"ML-0183\",\n        \"ML-0184\",\n        \"ML-0185\"\n      ],\n      \"validation_ids\": [\n        \"ML-0031\",\n        \"ML-0032\",\n        \"ML-0033\",\n        \"ML-0034\",\n        \"ML-0035\",\n        \"ML-0041\",\n        \"ML-0042\",\n        \"ML-0043\",\n        \"ML-0044\",\n        \"ML-0045\",\n        \"ML-0061\",\n        \"ML-0062\",\n        \"ML-0063\",\n        \"ML-0064\",\n        \"ML-0065\",\n        \"ML-0076\",\n        \"ML-0077\",\n        \"ML-0078\",\n        \"ML-0079\",\n        \"ML-0080\",\n        \"ML-0106\",\n        \"ML-0107\",\n        \"ML-0108\",\n        \"ML-0109\",\n        \"ML-0110\",\n        \"ML-0151\",\n        \"ML-0152\",\n        \"ML-0153\",\n        \"ML-0154\",\n        \"ML-0155\",\n        \"ML-0161\",\n        \"ML-0162\",\n        \"ML-0163\",\n        \"ML-0164\",\n        \"ML-0165\",\n        \"ML-0191\",\n        \"ML-0192\",\n        \"ML-0193\",\n        \"ML-0194\",\n        \"ML-0195\",\n        \"ML-0196\",\n        \"ML-0197\",\n        \"ML-0198\",\n        \"ML-0199\",\n        \"ML-0200\"\n      ]\n    }\n  ]\n}\n","ml/experiments/experiment-v1.0.0/local-2026-10-07/comparison.json":"{\n  \"experiment_version\": \"ticket-category-experiment-v1.0.0\",\n  \"phase\": \"development_only\",\n  \"training_executed\": true,\n  \"splits_created\": true,\n  \"provenance\": {\n    \"dataset\": {\n      \"dataset_version\": \"tickets-category-dataset-v1.0.0\",\n      \"dataset_commit\": \"9d3103991911b1b06b9faca6dce779013dae3036\",\n      \"dataset_sha256\": \"2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69\",\n      \"records\": 200,\n      \"distribution\": {\n        \"SOPORTE\": 50,\n        \"REDES\": 50,\n        \"INFRAESTRUCTURA\": 50,\n        \"BIOMEDICO\": 50\n      },\n      \"families\": 40,\n      \"leakage_groups\": 37,\n      \"is_synthetic\": true,\n      \"status\": \"DATASET_V1_VERIFIED\"\n    },\n    \"protocol_sha256_canonical_json\": \"c5eacbc7011971ad0f0ccbaf17b974866304386df5d7216869994a546fdb3959\",\n    \"splits_sha256_canonical_json\": \"e38ced2920e4fde8b13947155c58338bb9367213f0a2204fe2252936846c04ec\",\n    \"experiment_source_sha256_utf8_lf\": \"236d6c62a45b014eb74f5cf6b56ba305d948554a5aa4af87c75a0ae3eb9e37d0\",\n    \"preparation_source_sha256_utf8_lf\": \"e2ac1f6e1f2eb0da013e253869899ab4800c0e007c3a959d4c459a9dcaf06c98\",\n    \"requirements_sha256_utf8_lf\": \"38e73e22486a6b4346b321c94b95223ad53a53954e293e06af91c8fd69c4c86b\",\n    \"environment\": {\n      \"python\": \"3.12.14\",\n      \"pip_tool_version\": \"25.1.1\",\n      \"packages\": {\n        \"numpy\": \"2.2.6\",\n        \"scipy\": \"1.15.3\",\n        \"pandas\": \"2.2.3\",\n        \"scikit-learn\": \"1.6.1\",\n        \"matplotlib\": \"3.10.3\",\n        \"joblib\": \"1.5.1\",\n        \"threadpoolctl\": \"3.6.0\",\n        \"contourpy\": \"1.3.2\",\n        \"cycler\": \"0.12.1\",\n        \"fonttools\": \"4.58.0\",\n        \"kiwisolver\": \"1.4.8\",\n        \"packaging\": \"25.0\",\n        \"pillow\": \"11.2.1\",\n        \"pyparsing\": \"3.2.3\",\n        \"python-dateutil\": \"2.9.0.post0\",\n        \"pytz\": \"2025.2\",\n        \"six\": \"1.17.0\",\n        \"tzdata\": \"2025.2\"\n      }\n    },\n    \"source_mode\": \"local_snapshot\",\n    \"partition_summary\": {\n      \"development\": {\n        \"records\": 160,\n        \"groups\": 29,\n        \"distribution\": {\n          \"SOPORTE\": 40,\n          \"REDES\": 40,\n          \"BIOMEDICO\": 40,\n          \"INFRAESTRUCTURA\": 40\n        }\n      },\n      \"final_test\": {\n        \"records\": 40,\n        \"groups\": 8,\n        \"distribution\": {\n          \"BIOMEDICO\": 10,\n          \"REDES\": 10,\n          \"SOPORTE\": 10,\n          \"INFRAESTRUCTURA\": 10\n        }\n      },\n      \"folds\": [\n        {\n          \"fold\": 1,\n          \"training\": 120,\n          \"validation\": 40,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 10,\n            \"BIOMEDICO\": 10\n          }\n        },\n        {\n          \"fold\": 2,\n          \"training\": 125,\n          \"validation\": 35,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 10,\n            \"BIOMEDICO\": 5\n          }\n        },\n        {\n          \"fold\": 3,\n          \"training\": 120,\n          \"validation\": 40,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 15,\n            \"BIOMEDICO\": 5\n          }\n        },\n        {\n          \"fold\": 4,\n          \"training\": 115,\n          \"validation\": 45,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 5,\n            \"BIOMEDICO\": 20\n          }\n        }\n      ]\n    }\n  },\n  \"candidates\": [\n    {\n      \"id\": \"dummy-majority\",\n      \"algorithm\": \"DummyClassifier\",\n      \"spec\": {\n        \"id\": \"dummy-majority\",\n        \"algorithm\": \"DummyClassifier\",\n        \"strategy\": \"most_frequent\"\n      },\n      \"folds\": [\n        {\n          \"fold\": 1,\n          \"accuracy\": 0.25,\n          \"macro_f1\": 0.1,\n          \"weighted_f1\": 0.1,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.25,\n              \"recall\": 1.0,\n              \"f1-score\": 0.4,\n              \"support\": 10.0\n            },\n            \"accuracy\": 0.25,\n            \"macro avg\": {\n              \"precision\": 0.0625,\n              \"recall\": 0.25,\n              \"f1-score\": 0.1,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.0625,\n              \"recall\": 0.25,\n              \"f1-score\": 0.1,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              0,\n              0,\n              0,\n              10\n            ],\n            [\n              0,\n              0,\n              0,\n              10\n            ],\n            [\n              0,\n              0,\n              0,\n              10\n            ],\n            [\n              0,\n              0,\n              0,\n              10\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ]\n          ]\n        },\n        {\n          \"fold\": 2,\n          \"accuracy\": 0.14285714285714285,\n          \"macro_f1\": 0.0625,\n          \"weighted_f1\": 0.03571428571428571,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.14285714285714285,\n              \"recall\": 1.0,\n              \"f1-score\": 0.25,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.14285714285714285,\n            \"macro avg\": {\n              \"precision\": 0.03571428571428571,\n              \"recall\": 0.25,\n              \"f1-score\": 0.0625,\n              \"support\": 35.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.02040816326530612,\n              \"recall\": 0.14285714285714285,\n              \"f1-score\": 0.03571428571428571,\n              \"support\": 35.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              0,\n              0,\n              0,\n              10\n            ],\n            [\n              0,\n              0,\n              0,\n              10\n            ],\n            [\n              0,\n              0,\n              0,\n              10\n            ],\n            [\n              0,\n              0,\n              0,\n              5\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ]\n          ]\n        },\n        {\n          \"fold\": 3,\n          \"accuracy\": 0.125,\n          \"macro_f1\": 0.05555555555555555,\n          \"weighted_f1\": 0.02777777777777778,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 15.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.125,\n              \"recall\": 1.0,\n              \"f1-score\": 0.2222222222222222,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.125,\n            \"macro avg\": {\n              \"precision\": 0.03125,\n              \"recall\": 0.25,\n              \"f1-score\": 0.05555555555555555,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.015625,\n              \"recall\": 0.125,\n              \"f1-score\": 0.02777777777777778,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              0,\n              0,\n              0,\n              10\n            ],\n            [\n              0,\n              0,\n              0,\n              10\n            ],\n            [\n              0,\n              0,\n              0,\n              15\n            ],\n            [\n              0,\n              0,\n              0,\n              5\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ]\n          ]\n        },\n        {\n          \"fold\": 4,\n          \"accuracy\": 0.1111111111111111,\n          \"macro_f1\": 0.05,\n          \"weighted_f1\": 0.022222222222222223,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.1111111111111111,\n              \"recall\": 1.0,\n              \"f1-score\": 0.2,\n              \"support\": 5.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.0,\n              \"recall\": 0.0,\n              \"f1-score\": 0.0,\n              \"support\": 20.0\n            },\n            \"accuracy\": 0.1111111111111111,\n            \"macro avg\": {\n              \"precision\": 0.027777777777777776,\n              \"recall\": 0.25,\n              \"f1-score\": 0.05,\n              \"support\": 45.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.01234567901234568,\n              \"recall\": 0.1111111111111111,\n              \"f1-score\": 0.022222222222222223,\n              \"support\": 45.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              0,\n              0,\n              10,\n              0\n            ],\n            [\n              0,\n              0,\n              10,\n              0\n            ],\n            [\n              0,\n              0,\n              5,\n              0\n            ],\n            [\n              0,\n              0,\n              20,\n              0\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ]\n          ]\n        }\n      ],\n      \"mean_macro_f1\": 0.06701388888888889,\n      \"std_macro_f1\": 0.01955265031291818,\n      \"oof_metrics\": {\n        \"accuracy\": 0.15625,\n        \"macro_f1\": 0.09392789373814042,\n        \"weighted_f1\": 0.09392789373814041,\n        \"classification_report\": {\n          \"SOPORTE\": {\n            \"precision\": 0.0,\n            \"recall\": 0.0,\n            \"f1-score\": 0.0,\n            \"support\": 40.0\n          },\n          \"REDES\": {\n            \"precision\": 0.0,\n            \"recall\": 0.0,\n            \"f1-score\": 0.0,\n            \"support\": 40.0\n          },\n          \"INFRAESTRUCTURA\": {\n            \"precision\": 0.1111111111111111,\n            \"recall\": 0.125,\n            \"f1-score\": 0.11764705882352941,\n            \"support\": 40.0\n          },\n          \"BIOMEDICO\": {\n            \"precision\": 0.17391304347826086,\n            \"recall\": 0.5,\n            \"f1-score\": 0.25806451612903225,\n            \"support\": 40.0\n          },\n          \"accuracy\": 0.15625,\n          \"macro avg\": {\n            \"precision\": 0.07125603864734299,\n            \"recall\": 0.15625,\n            \"f1-score\": 0.09392789373814042,\n            \"support\": 160.0\n          },\n          \"weighted avg\": {\n            \"precision\": 0.071256038647343,\n            \"recall\": 0.15625,\n            \"f1-score\": 0.09392789373814041,\n            \"support\": 160.0\n          }\n        },\n        \"labels\": [\n          \"SOPORTE\",\n          \"REDES\",\n          \"INFRAESTRUCTURA\",\n          \"BIOMEDICO\"\n        ],\n        \"confusion_matrix\": [\n          [\n            0,\n            0,\n            10,\n            30\n          ],\n          [\n            0,\n            0,\n            10,\n            30\n          ],\n          [\n            0,\n            0,\n            5,\n            35\n          ],\n          [\n            0,\n            0,\n            20,\n            20\n          ]\n        ],\n        \"confusion_matrix_normalized\": [\n          [\n            0.0,\n            0.0,\n            0.25,\n            0.75\n          ],\n          [\n            0.0,\n            0.0,\n            0.25,\n            0.75\n          ],\n          [\n            0.0,\n            0.0,\n            0.125,\n            0.875\n          ],\n          [\n            0.0,\n            0.0,\n            0.5,\n            0.5\n          ]\n        ]\n      },\n      \"mean_fit_seconds\": 0.0011257749993092148,\n      \"mean_inference_seconds_per_record\": 1.023420633308498e-05,\n      \"score_kind\": \"baseline_no_confidence\"\n    },\n    {\n      \"id\": \"logreg-c1\",\n      \"algorithm\": \"LogisticRegression\",\n      \"spec\": {\n        \"id\": \"logreg-c1\",\n        \"algorithm\": \"LogisticRegression\",\n        \"C\": 1.0,\n        \"solver\": \"lbfgs\",\n        \"max_iter\": 2000\n      },\n      \"folds\": [\n        {\n          \"fold\": 1,\n          \"accuracy\": 0.95,\n          \"macro_f1\": 0.9484126984126984,\n          \"weighted_f1\": 0.9484126984126984,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.9090909090909091,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9523809523809523,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.9090909090909091,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9523809523809523,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 1.0,\n              \"recall\": 0.8,\n              \"f1-score\": 0.8888888888888888,\n              \"support\": 10.0\n            },\n            \"accuracy\": 0.95,\n            \"macro avg\": {\n              \"precision\": 0.9545454545454546,\n              \"recall\": 0.95,\n              \"f1-score\": 0.9484126984126984,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9545454545454545,\n              \"recall\": 0.95,\n              \"f1-score\": 0.9484126984126984,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              10,\n              0,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              10,\n              0\n            ],\n            [\n              1,\n              0,\n              1,\n              8\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              1.0,\n              0.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.1,\n              0.0,\n              0.1,\n              0.8\n            ]\n          ]\n        },\n        {\n          \"fold\": 2,\n          \"accuracy\": 0.8571428571428571,\n          \"macro_f1\": 0.8500667429443173,\n          \"weighted_f1\": 0.8524572300316008,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 1.0,\n              \"recall\": 0.6,\n              \"f1-score\": 0.75,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.7692307692307693,\n              \"recall\": 1.0,\n              \"f1-score\": 0.8695652173913043,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 1.0,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9473684210526315,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.7142857142857143,\n              \"recall\": 1.0,\n              \"f1-score\": 0.8333333333333334,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.8571428571428571,\n            \"macro avg\": {\n              \"precision\": 0.8708791208791209,\n              \"recall\": 0.875,\n              \"f1-score\": 0.8500667429443173,\n              \"support\": 35.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.8932496075353219,\n              \"recall\": 0.8571428571428571,\n              \"f1-score\": 0.8524572300316008,\n              \"support\": 35.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              6,\n              3,\n              0,\n              1\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              9,\n              1\n            ],\n            [\n              0,\n              0,\n              0,\n              5\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.6,\n              0.3,\n              0.0,\n              0.1\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.9,\n              0.1\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ]\n          ]\n        },\n        {\n          \"fold\": 3,\n          \"accuracy\": 0.9,\n          \"macro_f1\": 0.8666666666666667,\n          \"weighted_f1\": 0.9,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.8666666666666667,\n              \"recall\": 0.8666666666666667,\n              \"f1-score\": 0.8666666666666667,\n              \"support\": 15.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.6,\n              \"recall\": 0.6,\n              \"f1-score\": 0.6,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.9,\n            \"macro avg\": {\n              \"precision\": 0.8666666666666667,\n              \"recall\": 0.8666666666666667,\n              \"f1-score\": 0.8666666666666667,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              10,\n              0,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              13,\n              2\n            ],\n            [\n              0,\n              0,\n              2,\n              3\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              1.0,\n              0.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.8666666666666667,\n              0.13333333333333333\n            ],\n            [\n              0.0,\n              0.0,\n              0.4,\n              0.6\n            ]\n          ]\n        },\n        {\n          \"fold\": 4,\n          \"accuracy\": 0.5555555555555556,\n          \"macro_f1\": 0.5646026831785346,\n          \"weighted_f1\": 0.510583648664144,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.45,\n              \"recall\": 0.9,\n              \"f1-score\": 0.6,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.7777777777777778,\n              \"recall\": 0.7,\n              \"f1-score\": 0.7368421052631579,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.4166666666666667,\n              \"recall\": 1.0,\n              \"f1-score\": 0.5882352941176471,\n              \"support\": 5.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 1.0,\n              \"recall\": 0.2,\n              \"f1-score\": 0.3333333333333333,\n              \"support\": 20.0\n            },\n            \"accuracy\": 0.5555555555555556,\n            \"macro avg\": {\n              \"precision\": 0.6611111111111112,\n              \"recall\": 0.7000000000000001,\n              \"f1-score\": 0.5646026831785346,\n              \"support\": 45.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.7635802469135803,\n              \"recall\": 0.5555555555555556,\n              \"f1-score\": 0.510583648664144,\n              \"support\": 45.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              9,\n              1,\n              0,\n              0\n            ],\n            [\n              2,\n              7,\n              1,\n              0\n            ],\n            [\n              0,\n              0,\n              5,\n              0\n            ],\n            [\n              9,\n              1,\n              6,\n              4\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.9,\n              0.1,\n              0.0,\n              0.0\n            ],\n            [\n              0.2,\n              0.7,\n              0.1,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.45,\n              0.05,\n              0.3,\n              0.2\n            ]\n          ]\n        }\n      ],\n      \"mean_macro_f1\": 0.8074371978005542,\n      \"std_macro_f1\": 0.14505869619388773,\n      \"oof_metrics\": {\n        \"accuracy\": 0.80625,\n        \"macro_f1\": 0.7956528595458369,\n        \"weighted_f1\": 0.7956528595458369,\n        \"classification_report\": {\n          \"SOPORTE\": {\n            \"precision\": 0.7446808510638298,\n            \"recall\": 0.875,\n            \"f1-score\": 0.8045977011494253,\n            \"support\": 40.0\n          },\n          \"REDES\": {\n            \"precision\": 0.8809523809523809,\n            \"recall\": 0.925,\n            \"f1-score\": 0.9024390243902439,\n            \"support\": 40.0\n          },\n          \"INFRAESTRUCTURA\": {\n            \"precision\": 0.7872340425531915,\n            \"recall\": 0.925,\n            \"f1-score\": 0.8505747126436781,\n            \"support\": 40.0\n          },\n          \"BIOMEDICO\": {\n            \"precision\": 0.8333333333333334,\n            \"recall\": 0.5,\n            \"f1-score\": 0.625,\n            \"support\": 40.0\n          },\n          \"accuracy\": 0.80625,\n          \"macro avg\": {\n            \"precision\": 0.8115501519756839,\n            \"recall\": 0.80625,\n            \"f1-score\": 0.7956528595458369,\n            \"support\": 160.0\n          },\n          \"weighted avg\": {\n            \"precision\": 0.811550151975684,\n            \"recall\": 0.80625,\n            \"f1-score\": 0.7956528595458369,\n            \"support\": 160.0\n          }\n        },\n        \"labels\": [\n          \"SOPORTE\",\n          \"REDES\",\n          \"INFRAESTRUCTURA\",\n          \"BIOMEDICO\"\n        ],\n        \"confusion_matrix\": [\n          [\n            35,\n            4,\n            0,\n            1\n          ],\n          [\n            2,\n            37,\n            1,\n            0\n          ],\n          [\n            0,\n            0,\n            37,\n            3\n          ],\n          [\n            10,\n            1,\n            9,\n            20\n          ]\n        ],\n        \"confusion_matrix_normalized\": [\n          [\n            0.875,\n            0.1,\n            0.0,\n            0.025\n          ],\n          [\n            0.05,\n            0.925,\n            0.025,\n            0.0\n          ],\n          [\n            0.0,\n            0.0,\n            0.925,\n            0.075\n          ],\n          [\n            0.25,\n            0.025,\n            0.225,\n            0.5\n          ]\n        ]\n      },\n      \"mean_fit_seconds\": 0.12020790000133275,\n      \"mean_inference_seconds_per_record\": 0.0003115160714233276,\n      \"score_kind\": \"uncalibrated_probability\",\n      \"uncalibrated_probability_coverage_analysis\": [\n        {\n          \"threshold\": 0.25,\n          \"accepted\": 160,\n          \"coverage\": 1.0,\n          \"accepted_accuracy\": 0.80625,\n          \"abstentions_per_category\": {}\n        },\n        {\n          \"threshold\": 0.4,\n          \"accepted\": 89,\n          \"coverage\": 0.55625,\n          \"accepted_accuracy\": 0.9438202247191011,\n          \"abstentions_per_category\": {\n            \"SOPORTE\": 19,\n            \"REDES\": 11,\n            \"INFRAESTRUCTURA\": 14,\n            \"BIOMEDICO\": 27\n          }\n        },\n        {\n          \"threshold\": 0.55,\n          \"accepted\": 11,\n          \"coverage\": 0.06875,\n          \"accepted_accuracy\": 1.0,\n          \"abstentions_per_category\": {\n            \"SOPORTE\": 39,\n            \"REDES\": 34,\n            \"INFRAESTRUCTURA\": 37,\n            \"BIOMEDICO\": 39\n          }\n        },\n        {\n          \"threshold\": 0.7,\n          \"accepted\": 0,\n          \"coverage\": 0.0,\n          \"accepted_accuracy\": null,\n          \"abstentions_per_category\": {\n            \"SOPORTE\": 40,\n            \"REDES\": 40,\n            \"INFRAESTRUCTURA\": 40,\n            \"BIOMEDICO\": 40\n          }\n        },\n        {\n          \"threshold\": 0.85,\n          \"accepted\": 0,\n          \"coverage\": 0.0,\n          \"accepted_accuracy\": null,\n          \"abstentions_per_category\": {\n            \"SOPORTE\": 40,\n            \"REDES\": 40,\n            \"INFRAESTRUCTURA\": 40,\n            \"BIOMEDICO\": 40\n          }\n        }\n      ]\n    },\n    {\n      \"id\": \"logreg-c4\",\n      \"algorithm\": \"LogisticRegression\",\n      \"spec\": {\n        \"id\": \"logreg-c4\",\n        \"algorithm\": \"LogisticRegression\",\n        \"C\": 4.0,\n        \"solver\": \"lbfgs\",\n        \"max_iter\": 2000\n      },\n      \"folds\": [\n        {\n          \"fold\": 1,\n          \"accuracy\": 0.95,\n          \"macro_f1\": 0.9484126984126984,\n          \"weighted_f1\": 0.9484126984126984,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.9090909090909091,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9523809523809523,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.9090909090909091,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9523809523809523,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 1.0,\n              \"recall\": 0.8,\n              \"f1-score\": 0.8888888888888888,\n              \"support\": 10.0\n            },\n            \"accuracy\": 0.95,\n            \"macro avg\": {\n              \"precision\": 0.9545454545454546,\n              \"recall\": 0.95,\n              \"f1-score\": 0.9484126984126984,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9545454545454545,\n              \"recall\": 0.95,\n              \"f1-score\": 0.9484126984126984,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              10,\n              0,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              10,\n              0\n            ],\n            [\n              1,\n              0,\n              1,\n              8\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              1.0,\n              0.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.1,\n              0.0,\n              0.1,\n              0.8\n            ]\n          ]\n        },\n        {\n          \"fold\": 2,\n          \"accuracy\": 0.9142857142857143,\n          \"macro_f1\": 0.9136097820308346,\n          \"weighted_f1\": 0.9142553353079669,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 1.0,\n              \"recall\": 0.8,\n              \"f1-score\": 0.8888888888888888,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.8333333333333334,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9090909090909091,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 1.0,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9473684210526315,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.8333333333333334,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9090909090909091,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.9142857142857143,\n            \"macro avg\": {\n              \"precision\": 0.9166666666666667,\n              \"recall\": 0.925,\n              \"f1-score\": 0.9136097820308346,\n              \"support\": 35.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9285714285714286,\n              \"recall\": 0.9142857142857143,\n              \"f1-score\": 0.9142553353079669,\n              \"support\": 35.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              8,\n              2,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              9,\n              1\n            ],\n            [\n              0,\n              0,\n              0,\n              5\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.8,\n              0.2,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.9,\n              0.1\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ]\n          ]\n        },\n        {\n          \"fold\": 3,\n          \"accuracy\": 0.9,\n          \"macro_f1\": 0.8666666666666667,\n          \"weighted_f1\": 0.9,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.8666666666666667,\n              \"recall\": 0.8666666666666667,\n              \"f1-score\": 0.8666666666666667,\n              \"support\": 15.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.6,\n              \"recall\": 0.6,\n              \"f1-score\": 0.6,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.9,\n            \"macro avg\": {\n              \"precision\": 0.8666666666666667,\n              \"recall\": 0.8666666666666667,\n              \"f1-score\": 0.8666666666666667,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              10,\n              0,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              13,\n              2\n            ],\n            [\n              0,\n              0,\n              2,\n              3\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              1.0,\n              0.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.8666666666666667,\n              0.13333333333333333\n            ],\n            [\n              0.0,\n              0.0,\n              0.4,\n              0.6\n            ]\n          ]\n        },\n        {\n          \"fold\": 4,\n          \"accuracy\": 0.6222222222222222,\n          \"macro_f1\": 0.6424115845168477,\n          \"weighted_f1\": 0.6068937776540116,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.45,\n              \"recall\": 0.9,\n              \"f1-score\": 0.6,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.7777777777777778,\n              \"recall\": 0.7,\n              \"f1-score\": 0.7368421052631579,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.5555555555555556,\n              \"recall\": 1.0,\n              \"f1-score\": 0.7142857142857143,\n              \"support\": 5.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 1.0,\n              \"recall\": 0.35,\n              \"f1-score\": 0.5185185185185185,\n              \"support\": 20.0\n            },\n            \"accuracy\": 0.6222222222222222,\n            \"macro avg\": {\n              \"precision\": 0.6958333333333333,\n              \"recall\": 0.7375,\n              \"f1-score\": 0.6424115845168477,\n              \"support\": 45.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.7790123456790123,\n              \"recall\": 0.6222222222222222,\n              \"f1-score\": 0.6068937776540116,\n              \"support\": 45.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              9,\n              1,\n              0,\n              0\n            ],\n            [\n              2,\n              7,\n              1,\n              0\n            ],\n            [\n              0,\n              0,\n              5,\n              0\n            ],\n            [\n              9,\n              1,\n              3,\n              7\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.9,\n              0.1,\n              0.0,\n              0.0\n            ],\n            [\n              0.2,\n              0.7,\n              0.1,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.45,\n              0.05,\n              0.15,\n              0.35\n            ]\n          ]\n        }\n      ],\n      \"mean_macro_f1\": 0.8427751829067618,\n      \"std_macro_f1\": 0.11926147709061348,\n      \"oof_metrics\": {\n        \"accuracy\": 0.8375,\n        \"macro_f1\": 0.8307407497482404,\n        \"weighted_f1\": 0.8307407497482403,\n        \"classification_report\": {\n          \"SOPORTE\": {\n            \"precision\": 0.7551020408163265,\n            \"recall\": 0.925,\n            \"f1-score\": 0.8314606741573034,\n            \"support\": 40.0\n          },\n          \"REDES\": {\n            \"precision\": 0.9024390243902439,\n            \"recall\": 0.925,\n            \"f1-score\": 0.9135802469135802,\n            \"support\": 40.0\n          },\n          \"INFRAESTRUCTURA\": {\n            \"precision\": 0.8409090909090909,\n            \"recall\": 0.925,\n            \"f1-score\": 0.8809523809523809,\n            \"support\": 40.0\n          },\n          \"BIOMEDICO\": {\n            \"precision\": 0.8846153846153846,\n            \"recall\": 0.575,\n            \"f1-score\": 0.696969696969697,\n            \"support\": 40.0\n          },\n          \"accuracy\": 0.8375,\n          \"macro avg\": {\n            \"precision\": 0.8457663851827615,\n            \"recall\": 0.8375000000000001,\n            \"f1-score\": 0.8307407497482404,\n            \"support\": 160.0\n          },\n          \"weighted avg\": {\n            \"precision\": 0.8457663851827615,\n            \"recall\": 0.8375,\n            \"f1-score\": 0.8307407497482403,\n            \"support\": 160.0\n          }\n        },\n        \"labels\": [\n          \"SOPORTE\",\n          \"REDES\",\n          \"INFRAESTRUCTURA\",\n          \"BIOMEDICO\"\n        ],\n        \"confusion_matrix\": [\n          [\n            37,\n            3,\n            0,\n            0\n          ],\n          [\n            2,\n            37,\n            1,\n            0\n          ],\n          [\n            0,\n            0,\n            37,\n            3\n          ],\n          [\n            10,\n            1,\n            6,\n            23\n          ]\n        ],\n        \"confusion_matrix_normalized\": [\n          [\n            0.925,\n            0.075,\n            0.0,\n            0.0\n          ],\n          [\n            0.05,\n            0.925,\n            0.025,\n            0.0\n          ],\n          [\n            0.0,\n            0.0,\n            0.925,\n            0.075\n          ],\n          [\n            0.25,\n            0.025,\n            0.15,\n            0.575\n          ]\n        ]\n      },\n      \"mean_fit_seconds\": 0.14358970000103,\n      \"mean_inference_seconds_per_record\": 0.0003195837500387579,\n      \"score_kind\": \"uncalibrated_probability\",\n      \"uncalibrated_probability_coverage_analysis\": [\n        {\n          \"threshold\": 0.25,\n          \"accepted\": 160,\n          \"coverage\": 1.0,\n          \"accepted_accuracy\": 0.8375,\n          \"abstentions_per_category\": {}\n        },\n        {\n          \"threshold\": 0.4,\n          \"accepted\": 123,\n          \"coverage\": 0.76875,\n          \"accepted_accuracy\": 0.9105691056910569,\n          \"abstentions_per_category\": {\n            \"SOPORTE\": 6,\n            \"REDES\": 2,\n            \"INFRAESTRUCTURA\": 8,\n            \"BIOMEDICO\": 21\n          }\n        },\n        {\n          \"threshold\": 0.55,\n          \"accepted\": 65,\n          \"coverage\": 0.40625,\n          \"accepted_accuracy\": 0.9538461538461539,\n          \"abstentions_per_category\": {\n            \"SOPORTE\": 25,\n            \"REDES\": 18,\n            \"INFRAESTRUCTURA\": 20,\n            \"BIOMEDICO\": 32\n          }\n        },\n        {\n          \"threshold\": 0.7,\n          \"accepted\": 15,\n          \"coverage\": 0.09375,\n          \"accepted_accuracy\": 1.0,\n          \"abstentions_per_category\": {\n            \"SOPORTE\": 38,\n            \"REDES\": 32,\n            \"INFRAESTRUCTURA\": 36,\n            \"BIOMEDICO\": 39\n          }\n        },\n        {\n          \"threshold\": 0.85,\n          \"accepted\": 0,\n          \"coverage\": 0.0,\n          \"accepted_accuracy\": null,\n          \"abstentions_per_category\": {\n            \"SOPORTE\": 40,\n            \"REDES\": 40,\n            \"INFRAESTRUCTURA\": 40,\n            \"BIOMEDICO\": 40\n          }\n        }\n      ]\n    },\n    {\n      \"id\": \"linear-svc-c1\",\n      \"algorithm\": \"LinearSVC\",\n      \"spec\": {\n        \"id\": \"linear-svc-c1\",\n        \"algorithm\": \"LinearSVC\",\n        \"C\": 1.0,\n        \"dual\": \"auto\",\n        \"max_iter\": 10000\n      },\n      \"folds\": [\n        {\n          \"fold\": 1,\n          \"accuracy\": 0.975,\n          \"macro_f1\": 0.974937343358396,\n          \"weighted_f1\": 0.974937343358396,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.9090909090909091,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9523809523809523,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 1.0,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9473684210526315,\n              \"support\": 10.0\n            },\n            \"accuracy\": 0.975,\n            \"macro avg\": {\n              \"precision\": 0.9772727272727273,\n              \"recall\": 0.975,\n              \"f1-score\": 0.974937343358396,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9772727272727273,\n              \"recall\": 0.975,\n              \"f1-score\": 0.974937343358396,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              10,\n              0,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              10,\n              0\n            ],\n            [\n              1,\n              0,\n              0,\n              9\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              1.0,\n              0.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.1,\n              0.0,\n              0.0,\n              0.9\n            ]\n          ]\n        },\n        {\n          \"fold\": 2,\n          \"accuracy\": 0.9142857142857143,\n          \"macro_f1\": 0.9136097820308346,\n          \"weighted_f1\": 0.9142553353079669,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 1.0,\n              \"recall\": 0.8,\n              \"f1-score\": 0.8888888888888888,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.8333333333333334,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9090909090909091,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 1.0,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9473684210526315,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.8333333333333334,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9090909090909091,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.9142857142857143,\n            \"macro avg\": {\n              \"precision\": 0.9166666666666667,\n              \"recall\": 0.925,\n              \"f1-score\": 0.9136097820308346,\n              \"support\": 35.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9285714285714286,\n              \"recall\": 0.9142857142857143,\n              \"f1-score\": 0.9142553353079669,\n              \"support\": 35.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              8,\n              2,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              9,\n              1\n            ],\n            [\n              0,\n              0,\n              0,\n              5\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.8,\n              0.2,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.9,\n              0.1\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ]\n          ]\n        },\n        {\n          \"fold\": 3,\n          \"accuracy\": 0.9,\n          \"macro_f1\": 0.8666666666666667,\n          \"weighted_f1\": 0.9,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.8666666666666667,\n              \"recall\": 0.8666666666666667,\n              \"f1-score\": 0.8666666666666667,\n              \"support\": 15.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.6,\n              \"recall\": 0.6,\n              \"f1-score\": 0.6,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.9,\n            \"macro avg\": {\n              \"precision\": 0.8666666666666667,\n              \"recall\": 0.8666666666666667,\n              \"f1-score\": 0.8666666666666667,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              10,\n              0,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              13,\n              2\n            ],\n            [\n              0,\n              0,\n              2,\n              3\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              1.0,\n              0.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.8666666666666667,\n              0.13333333333333333\n            ],\n            [\n              0.0,\n              0.0,\n              0.4,\n              0.6\n            ]\n          ]\n        },\n        {\n          \"fold\": 4,\n          \"accuracy\": 0.7111111111111111,\n          \"macro_f1\": 0.7206042401288582,\n          \"weighted_f1\": 0.7127731100566416,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.5294117647058824,\n              \"recall\": 0.9,\n              \"f1-score\": 0.6666666666666666,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.7777777777777778,\n              \"recall\": 0.7,\n              \"f1-score\": 0.7368421052631579,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.625,\n              \"recall\": 1.0,\n              \"f1-score\": 0.7692307692307693,\n              \"support\": 5.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 1.0,\n              \"recall\": 0.55,\n              \"f1-score\": 0.7096774193548387,\n              \"support\": 20.0\n            },\n            \"accuracy\": 0.7111111111111111,\n            \"macro avg\": {\n              \"precision\": 0.733047385620915,\n              \"recall\": 0.7875000000000001,\n              \"f1-score\": 0.7206042401288582,\n              \"support\": 45.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.8043754538852578,\n              \"recall\": 0.7111111111111111,\n              \"f1-score\": 0.7127731100566416,\n              \"support\": 45.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              9,\n              1,\n              0,\n              0\n            ],\n            [\n              2,\n              7,\n              1,\n              0\n            ],\n            [\n              0,\n              0,\n              5,\n              0\n            ],\n            [\n              6,\n              1,\n              2,\n              11\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.9,\n              0.1,\n              0.0,\n              0.0\n            ],\n            [\n              0.2,\n              0.7,\n              0.1,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.3,\n              0.05,\n              0.1,\n              0.55\n            ]\n          ]\n        }\n      ],\n      \"mean_macro_f1\": 0.8689545080461889,\n      \"std_macro_f1\": 0.09386092249277633,\n      \"oof_metrics\": {\n        \"accuracy\": 0.86875,\n        \"macro_f1\": 0.8663041954872727,\n        \"weighted_f1\": 0.8663041954872727,\n        \"classification_report\": {\n          \"SOPORTE\": {\n            \"precision\": 0.8043478260869565,\n            \"recall\": 0.925,\n            \"f1-score\": 0.8604651162790697,\n            \"support\": 40.0\n          },\n          \"REDES\": {\n            \"precision\": 0.9024390243902439,\n            \"recall\": 0.925,\n            \"f1-score\": 0.9135802469135802,\n            \"support\": 40.0\n          },\n          \"INFRAESTRUCTURA\": {\n            \"precision\": 0.8809523809523809,\n            \"recall\": 0.925,\n            \"f1-score\": 0.9024390243902439,\n            \"support\": 40.0\n          },\n          \"BIOMEDICO\": {\n            \"precision\": 0.9032258064516129,\n            \"recall\": 0.7,\n            \"f1-score\": 0.7887323943661971,\n            \"support\": 40.0\n          },\n          \"accuracy\": 0.86875,\n          \"macro avg\": {\n            \"precision\": 0.8727412594702986,\n            \"recall\": 0.8687500000000001,\n            \"f1-score\": 0.8663041954872727,\n            \"support\": 160.0\n          },\n          \"weighted avg\": {\n            \"precision\": 0.8727412594702987,\n            \"recall\": 0.86875,\n            \"f1-score\": 0.8663041954872727,\n            \"support\": 160.0\n          }\n        },\n        \"labels\": [\n          \"SOPORTE\",\n          \"REDES\",\n          \"INFRAESTRUCTURA\",\n          \"BIOMEDICO\"\n        ],\n        \"confusion_matrix\": [\n          [\n            37,\n            3,\n            0,\n            0\n          ],\n          [\n            2,\n            37,\n            1,\n            0\n          ],\n          [\n            0,\n            0,\n            37,\n            3\n          ],\n          [\n            7,\n            1,\n            4,\n            28\n          ]\n        ],\n        \"confusion_matrix_normalized\": [\n          [\n            0.925,\n            0.075,\n            0.0,\n            0.0\n          ],\n          [\n            0.05,\n            0.925,\n            0.025,\n            0.0\n          ],\n          [\n            0.0,\n            0.0,\n            0.925,\n            0.075\n          ],\n          [\n            0.175,\n            0.025,\n            0.1,\n            0.7\n          ]\n        ]\n      },\n      \"mean_fit_seconds\": 0.0579552250001143,\n      \"mean_inference_seconds_per_record\": 0.00032259148810919125,\n      \"score_kind\": \"raw_decision_margin_not_probability\"\n    },\n    {\n      \"id\": \"linear-svc-c4\",\n      \"algorithm\": \"LinearSVC\",\n      \"spec\": {\n        \"id\": \"linear-svc-c4\",\n        \"algorithm\": \"LinearSVC\",\n        \"C\": 4.0,\n        \"dual\": \"auto\",\n        \"max_iter\": 10000\n      },\n      \"folds\": [\n        {\n          \"fold\": 1,\n          \"accuracy\": 0.975,\n          \"macro_f1\": 0.974937343358396,\n          \"weighted_f1\": 0.974937343358396,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.9090909090909091,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9523809523809523,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 1.0,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9473684210526315,\n              \"support\": 10.0\n            },\n            \"accuracy\": 0.975,\n            \"macro avg\": {\n              \"precision\": 0.9772727272727273,\n              \"recall\": 0.975,\n              \"f1-score\": 0.974937343358396,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9772727272727273,\n              \"recall\": 0.975,\n              \"f1-score\": 0.974937343358396,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              10,\n              0,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              10,\n              0\n            ],\n            [\n              1,\n              0,\n              0,\n              9\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              1.0,\n              0.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.1,\n              0.0,\n              0.0,\n              0.9\n            ]\n          ]\n        },\n        {\n          \"fold\": 2,\n          \"accuracy\": 0.9142857142857143,\n          \"macro_f1\": 0.9136097820308346,\n          \"weighted_f1\": 0.9142553353079669,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 1.0,\n              \"recall\": 0.8,\n              \"f1-score\": 0.8888888888888888,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.8333333333333334,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9090909090909091,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 1.0,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9473684210526315,\n              \"support\": 10.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.8333333333333334,\n              \"recall\": 1.0,\n              \"f1-score\": 0.9090909090909091,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.9142857142857143,\n            \"macro avg\": {\n              \"precision\": 0.9166666666666667,\n              \"recall\": 0.925,\n              \"f1-score\": 0.9136097820308346,\n              \"support\": 35.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9285714285714286,\n              \"recall\": 0.9142857142857143,\n              \"f1-score\": 0.9142553353079669,\n              \"support\": 35.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              8,\n              2,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              9,\n              1\n            ],\n            [\n              0,\n              0,\n              0,\n              5\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.8,\n              0.2,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.9,\n              0.1\n            ],\n            [\n              0.0,\n              0.0,\n              0.0,\n              1.0\n            ]\n          ]\n        },\n        {\n          \"fold\": 3,\n          \"accuracy\": 0.9,\n          \"macro_f1\": 0.8666666666666667,\n          \"weighted_f1\": 0.9,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 1.0,\n              \"recall\": 1.0,\n              \"f1-score\": 1.0,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.8666666666666667,\n              \"recall\": 0.8666666666666667,\n              \"f1-score\": 0.8666666666666667,\n              \"support\": 15.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 0.6,\n              \"recall\": 0.6,\n              \"f1-score\": 0.6,\n              \"support\": 5.0\n            },\n            \"accuracy\": 0.9,\n            \"macro avg\": {\n              \"precision\": 0.8666666666666667,\n              \"recall\": 0.8666666666666667,\n              \"f1-score\": 0.8666666666666667,\n              \"support\": 40.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.9,\n              \"recall\": 0.9,\n              \"f1-score\": 0.9,\n              \"support\": 40.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              10,\n              0,\n              0,\n              0\n            ],\n            [\n              0,\n              10,\n              0,\n              0\n            ],\n            [\n              0,\n              0,\n              13,\n              2\n            ],\n            [\n              0,\n              0,\n              2,\n              3\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              1.0,\n              0.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              1.0,\n              0.0,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              0.8666666666666667,\n              0.13333333333333333\n            ],\n            [\n              0.0,\n              0.0,\n              0.4,\n              0.6\n            ]\n          ]\n        },\n        {\n          \"fold\": 4,\n          \"accuracy\": 0.7111111111111111,\n          \"macro_f1\": 0.7132782328028509,\n          \"weighted_f1\": 0.7123661096496411,\n          \"classification_report\": {\n            \"SOPORTE\": {\n              \"precision\": 0.5625,\n              \"recall\": 0.9,\n              \"f1-score\": 0.6923076923076923,\n              \"support\": 10.0\n            },\n            \"REDES\": {\n              \"precision\": 0.7777777777777778,\n              \"recall\": 0.7,\n              \"f1-score\": 0.7368421052631579,\n              \"support\": 10.0\n            },\n            \"INFRAESTRUCTURA\": {\n              \"precision\": 0.5555555555555556,\n              \"recall\": 1.0,\n              \"f1-score\": 0.7142857142857143,\n              \"support\": 5.0\n            },\n            \"BIOMEDICO\": {\n              \"precision\": 1.0,\n              \"recall\": 0.55,\n              \"f1-score\": 0.7096774193548387,\n              \"support\": 20.0\n            },\n            \"accuracy\": 0.7111111111111111,\n            \"macro avg\": {\n              \"precision\": 0.7239583333333333,\n              \"recall\": 0.7875000000000001,\n              \"f1-score\": 0.7132782328028509,\n              \"support\": 45.0\n            },\n            \"weighted avg\": {\n              \"precision\": 0.8040123456790124,\n              \"recall\": 0.7111111111111111,\n              \"f1-score\": 0.7123661096496411,\n              \"support\": 45.0\n            }\n          },\n          \"labels\": [\n            \"SOPORTE\",\n            \"REDES\",\n            \"INFRAESTRUCTURA\",\n            \"BIOMEDICO\"\n          ],\n          \"confusion_matrix\": [\n            [\n              9,\n              1,\n              0,\n              0\n            ],\n            [\n              1,\n              7,\n              2,\n              0\n            ],\n            [\n              0,\n              0,\n              5,\n              0\n            ],\n            [\n              6,\n              1,\n              2,\n              11\n            ]\n          ],\n          \"confusion_matrix_normalized\": [\n            [\n              0.9,\n              0.1,\n              0.0,\n              0.0\n            ],\n            [\n              0.1,\n              0.7,\n              0.2,\n              0.0\n            ],\n            [\n              0.0,\n              0.0,\n              1.0,\n              0.0\n            ],\n            [\n              0.3,\n              0.05,\n              0.1,\n              0.55\n            ]\n          ]\n        }\n      ],\n      \"mean_macro_f1\": 0.8671230062146871,\n      \"std_macro_f1\": 0.09676437124716981,\n      \"oof_metrics\": {\n        \"accuracy\": 0.86875,\n        \"macro_f1\": 0.866116785408534,\n        \"weighted_f1\": 0.866116785408534,\n        \"classification_report\": {\n          \"SOPORTE\": {\n            \"precision\": 0.8222222222222222,\n            \"recall\": 0.925,\n            \"f1-score\": 0.8705882352941177,\n            \"support\": 40.0\n          },\n          \"REDES\": {\n            \"precision\": 0.9024390243902439,\n            \"recall\": 0.925,\n            \"f1-score\": 0.9135802469135802,\n            \"support\": 40.0\n          },\n          \"INFRAESTRUCTURA\": {\n            \"precision\": 0.8604651162790697,\n            \"recall\": 0.925,\n            \"f1-score\": 0.891566265060241,\n            \"support\": 40.0\n          },\n          \"BIOMEDICO\": {\n            \"precision\": 0.9032258064516129,\n            \"recall\": 0.7,\n            \"f1-score\": 0.7887323943661971,\n            \"support\": 40.0\n          },\n          \"accuracy\": 0.86875,\n          \"macro avg\": {\n            \"precision\": 0.8720880423357873,\n            \"recall\": 0.8687500000000001,\n            \"f1-score\": 0.866116785408534,\n            \"support\": 160.0\n          },\n          \"weighted avg\": {\n            \"precision\": 0.8720880423357871,\n            \"recall\": 0.86875,\n            \"f1-score\": 0.866116785408534,\n            \"support\": 160.0\n          }\n        },\n        \"labels\": [\n          \"SOPORTE\",\n          \"REDES\",\n          \"INFRAESTRUCTURA\",\n          \"BIOMEDICO\"\n        ],\n        \"confusion_matrix\": [\n          [\n            37,\n            3,\n            0,\n            0\n          ],\n          [\n            1,\n            37,\n            2,\n            0\n          ],\n          [\n            0,\n            0,\n            37,\n            3\n          ],\n          [\n            7,\n            1,\n            4,\n            28\n          ]\n        ],\n        \"confusion_matrix_normalized\": [\n          [\n            0.925,\n            0.075,\n            0.0,\n            0.0\n          ],\n          [\n            0.025,\n            0.925,\n            0.05,\n            0.0\n          ],\n          [\n            0.0,\n            0.0,\n            0.925,\n            0.075\n          ],\n          [\n            0.175,\n            0.025,\n            0.1,\n            0.7\n          ]\n        ]\n      },\n      \"mean_fit_seconds\": 0.05517295000208833,\n      \"mean_inference_seconds_per_record\": 0.00032068156751270254,\n      \"score_kind\": \"raw_decision_margin_not_probability\"\n    }\n  ]\n}\n","ml/experiments/experiment-v1.0.0/local-2026-10-07/selection.json":"{\n  \"experiment_version\": \"ticket-category-experiment-v1.0.0\",\n  \"provenance\": {\n    \"dataset\": {\n      \"dataset_version\": \"tickets-category-dataset-v1.0.0\",\n      \"dataset_commit\": \"9d3103991911b1b06b9faca6dce779013dae3036\",\n      \"dataset_sha256\": \"2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69\",\n      \"records\": 200,\n      \"distribution\": {\n        \"SOPORTE\": 50,\n        \"REDES\": 50,\n        \"INFRAESTRUCTURA\": 50,\n        \"BIOMEDICO\": 50\n      },\n      \"families\": 40,\n      \"leakage_groups\": 37,\n      \"is_synthetic\": true,\n      \"status\": \"DATASET_V1_VERIFIED\"\n    },\n    \"protocol_sha256_canonical_json\": \"c5eacbc7011971ad0f0ccbaf17b974866304386df5d7216869994a546fdb3959\",\n    \"splits_sha256_canonical_json\": \"e38ced2920e4fde8b13947155c58338bb9367213f0a2204fe2252936846c04ec\",\n    \"experiment_source_sha256_utf8_lf\": \"236d6c62a45b014eb74f5cf6b56ba305d948554a5aa4af87c75a0ae3eb9e37d0\",\n    \"preparation_source_sha256_utf8_lf\": \"e2ac1f6e1f2eb0da013e253869899ab4800c0e007c3a959d4c459a9dcaf06c98\",\n    \"requirements_sha256_utf8_lf\": \"38e73e22486a6b4346b321c94b95223ad53a53954e293e06af91c8fd69c4c86b\",\n    \"environment\": {\n      \"python\": \"3.12.14\",\n      \"pip_tool_version\": \"25.1.1\",\n      \"packages\": {\n        \"numpy\": \"2.2.6\",\n        \"scipy\": \"1.15.3\",\n        \"pandas\": \"2.2.3\",\n        \"scikit-learn\": \"1.6.1\",\n        \"matplotlib\": \"3.10.3\",\n        \"joblib\": \"1.5.1\",\n        \"threadpoolctl\": \"3.6.0\",\n        \"contourpy\": \"1.3.2\",\n        \"cycler\": \"0.12.1\",\n        \"fonttools\": \"4.58.0\",\n        \"kiwisolver\": \"1.4.8\",\n        \"packaging\": \"25.0\",\n        \"pillow\": \"11.2.1\",\n        \"pyparsing\": \"3.2.3\",\n        \"python-dateutil\": \"2.9.0.post0\",\n        \"pytz\": \"2025.2\",\n        \"six\": \"1.17.0\",\n        \"tzdata\": \"2025.2\"\n      }\n    },\n    \"source_mode\": \"local_snapshot\",\n    \"partition_summary\": {\n      \"development\": {\n        \"records\": 160,\n        \"groups\": 29,\n        \"distribution\": {\n          \"SOPORTE\": 40,\n          \"REDES\": 40,\n          \"BIOMEDICO\": 40,\n          \"INFRAESTRUCTURA\": 40\n        }\n      },\n      \"final_test\": {\n        \"records\": 40,\n        \"groups\": 8,\n        \"distribution\": {\n          \"BIOMEDICO\": 10,\n          \"REDES\": 10,\n          \"SOPORTE\": 10,\n          \"INFRAESTRUCTURA\": 10\n        }\n      },\n      \"folds\": [\n        {\n          \"fold\": 1,\n          \"training\": 120,\n          \"validation\": 40,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 10,\n            \"BIOMEDICO\": 10\n          }\n        },\n        {\n          \"fold\": 2,\n          \"training\": 125,\n          \"validation\": 35,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 10,\n            \"BIOMEDICO\": 5\n          }\n        },\n        {\n          \"fold\": 3,\n          \"training\": 120,\n          \"validation\": 40,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 15,\n            \"BIOMEDICO\": 5\n          }\n        },\n        {\n          \"fold\": 4,\n          \"training\": 115,\n          \"validation\": 45,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 5,\n            \"BIOMEDICO\": 20\n          }\n        }\n      ]\n    }\n  },\n  \"decision\": {\n    \"candidate_id\": \"linear-svc-c1\",\n    \"candidate_spec\": {\n      \"id\": \"linear-svc-c1\",\n      \"algorithm\": \"LinearSVC\",\n      \"C\": 1.0,\n      \"dual\": \"auto\",\n      \"max_iter\": 10000\n    },\n    \"selection_source\": \"development_grouped_cv_only\",\n    \"cv_acceptance_passed\": true,\n    \"best_mean_candidate_id\": \"linear-svc-c1\",\n    \"near_tie_candidates\": [\n      \"linear-svc-c1\",\n      \"linear-svc-c4\"\n    ],\n    \"chosen_mean_macro_f1\": 0.8689545080461889,\n    \"chosen_std_macro_f1\": 0.09386092249277633,\n    \"baseline_mean_macro_f1\": 0.06701388888888889,\n    \"reason\": \"Eligibility gates first; CV mean/variability and predeclared algorithm/lower-C preference\"\n  },\n  \"comparison_sha256_canonical_json\": \"3fff280b7b3e3f9bd2c4ba09bcce85559d5e0f7606e5c7953f553d848613d038\",\n  \"final_test_used_for_selection\": false,\n  \"selection_sha256_canonical_json\": \"64e14be457ebe135f05fdf94bf5c6acbec787f3a78d87a044ef20c8a91b440d0\"\n}\n","ml/experiments/experiment-v1.0.0/local-2026-10-07/final-metrics.json":"{\n  \"experiment_version\": \"ticket-category-experiment-v1.0.0\",\n  \"phase\": \"final_test\",\n  \"provenance\": {\n    \"dataset\": {\n      \"dataset_version\": \"tickets-category-dataset-v1.0.0\",\n      \"dataset_commit\": \"9d3103991911b1b06b9faca6dce779013dae3036\",\n      \"dataset_sha256\": \"2f571a40f063c4cce267352582f0bf78d2008d09ddc617ab54c7f8dd5ca21a69\",\n      \"records\": 200,\n      \"distribution\": {\n        \"SOPORTE\": 50,\n        \"REDES\": 50,\n        \"INFRAESTRUCTURA\": 50,\n        \"BIOMEDICO\": 50\n      },\n      \"families\": 40,\n      \"leakage_groups\": 37,\n      \"is_synthetic\": true,\n      \"status\": \"DATASET_V1_VERIFIED\"\n    },\n    \"protocol_sha256_canonical_json\": \"c5eacbc7011971ad0f0ccbaf17b974866304386df5d7216869994a546fdb3959\",\n    \"splits_sha256_canonical_json\": \"e38ced2920e4fde8b13947155c58338bb9367213f0a2204fe2252936846c04ec\",\n    \"experiment_source_sha256_utf8_lf\": \"236d6c62a45b014eb74f5cf6b56ba305d948554a5aa4af87c75a0ae3eb9e37d0\",\n    \"preparation_source_sha256_utf8_lf\": \"e2ac1f6e1f2eb0da013e253869899ab4800c0e007c3a959d4c459a9dcaf06c98\",\n    \"requirements_sha256_utf8_lf\": \"38e73e22486a6b4346b321c94b95223ad53a53954e293e06af91c8fd69c4c86b\",\n    \"environment\": {\n      \"python\": \"3.12.14\",\n      \"pip_tool_version\": \"25.1.1\",\n      \"packages\": {\n        \"numpy\": \"2.2.6\",\n        \"scipy\": \"1.15.3\",\n        \"pandas\": \"2.2.3\",\n        \"scikit-learn\": \"1.6.1\",\n        \"matplotlib\": \"3.10.3\",\n        \"joblib\": \"1.5.1\",\n        \"threadpoolctl\": \"3.6.0\",\n        \"contourpy\": \"1.3.2\",\n        \"cycler\": \"0.12.1\",\n        \"fonttools\": \"4.58.0\",\n        \"kiwisolver\": \"1.4.8\",\n        \"packaging\": \"25.0\",\n        \"pillow\": \"11.2.1\",\n        \"pyparsing\": \"3.2.3\",\n        \"python-dateutil\": \"2.9.0.post0\",\n        \"pytz\": \"2025.2\",\n        \"six\": \"1.17.0\",\n        \"tzdata\": \"2025.2\"\n      }\n    },\n    \"source_mode\": \"local_snapshot\",\n    \"partition_summary\": {\n      \"development\": {\n        \"records\": 160,\n        \"groups\": 29,\n        \"distribution\": {\n          \"BIOMEDICO\": 40,\n          \"INFRAESTRUCTURA\": 40,\n          \"REDES\": 40,\n          \"SOPORTE\": 40\n        }\n      },\n      \"final_test\": {\n        \"records\": 40,\n        \"groups\": 8,\n        \"distribution\": {\n          \"REDES\": 10,\n          \"SOPORTE\": 10,\n          \"BIOMEDICO\": 10,\n          \"INFRAESTRUCTURA\": 10\n        }\n      },\n      \"folds\": [\n        {\n          \"fold\": 1,\n          \"training\": 120,\n          \"validation\": 40,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 10,\n            \"BIOMEDICO\": 10\n          }\n        },\n        {\n          \"fold\": 2,\n          \"training\": 125,\n          \"validation\": 35,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 10,\n            \"BIOMEDICO\": 5\n          }\n        },\n        {\n          \"fold\": 3,\n          \"training\": 120,\n          \"validation\": 40,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 15,\n            \"BIOMEDICO\": 5\n          }\n        },\n        {\n          \"fold\": 4,\n          \"training\": 115,\n          \"validation\": 45,\n          \"validation_distribution\": {\n            \"SOPORTE\": 10,\n            \"REDES\": 10,\n            \"INFRAESTRUCTURA\": 5,\n            \"BIOMEDICO\": 20\n          }\n        }\n      ]\n    }\n  },\n  \"training_executed\": true,\n  \"splits_created\": true,\n  \"selection_sha256_canonical_json\": \"64e14be457ebe135f05fdf94bf5c6acbec787f3a78d87a044ef20c8a91b440d0\",\n  \"candidate_id\": \"linear-svc-c1\",\n  \"score_kind\": \"raw_decision_margin_not_probability\",\n  \"metrics\": {\n    \"accuracy\": 1.0,\n    \"macro_f1\": 1.0,\n    \"weighted_f1\": 1.0,\n    \"classification_report\": {\n      \"SOPORTE\": {\n        \"precision\": 1.0,\n        \"recall\": 1.0,\n        \"f1-score\": 1.0,\n        \"support\": 10.0\n      },\n      \"REDES\": {\n        \"precision\": 1.0,\n        \"recall\": 1.0,\n        \"f1-score\": 1.0,\n        \"support\": 10.0\n      },\n      \"INFRAESTRUCTURA\": {\n        \"precision\": 1.0,\n        \"recall\": 1.0,\n        \"f1-score\": 1.0,\n        \"support\": 10.0\n      },\n      \"BIOMEDICO\": {\n        \"precision\": 1.0,\n        \"recall\": 1.0,\n        \"f1-score\": 1.0,\n        \"support\": 10.0\n      },\n      \"accuracy\": 1.0,\n      \"macro avg\": {\n        \"precision\": 1.0,\n        \"recall\": 1.0,\n        \"f1-score\": 1.0,\n        \"support\": 40.0\n      },\n      \"weighted avg\": {\n        \"precision\": 1.0,\n        \"recall\": 1.0,\n        \"f1-score\": 1.0,\n        \"support\": 40.0\n      }\n    },\n    \"labels\": [\n      \"SOPORTE\",\n      \"REDES\",\n      \"INFRAESTRUCTURA\",\n      \"BIOMEDICO\"\n    ],\n    \"confusion_matrix\": [\n      [\n        10,\n        0,\n        0,\n        0\n      ],\n      [\n        0,\n        10,\n        0,\n        0\n      ],\n      [\n        0,\n        0,\n        10,\n        0\n      ],\n      [\n        0,\n        0,\n        0,\n        10\n      ]\n    ],\n    \"confusion_matrix_normalized\": [\n      [\n        1.0,\n        0.0,\n        0.0,\n        0.0\n      ],\n      [\n        0.0,\n        1.0,\n        0.0,\n        0.0\n      ],\n      [\n        0.0,\n        0.0,\n        1.0,\n        0.0\n      ],\n      [\n        0.0,\n        0.0,\n        0.0,\n        1.0\n      ]\n    ]\n  },\n  \"baseline_metrics\": {\n    \"accuracy\": 0.25,\n    \"macro_f1\": 0.1,\n    \"weighted_f1\": 0.1,\n    \"classification_report\": {\n      \"SOPORTE\": {\n        \"precision\": 0.0,\n        \"recall\": 0.0,\n        \"f1-score\": 0.0,\n        \"support\": 10.0\n      },\n      \"REDES\": {\n        \"precision\": 0.0,\n        \"recall\": 0.0,\n        \"f1-score\": 0.0,\n        \"support\": 10.0\n      },\n      \"INFRAESTRUCTURA\": {\n        \"precision\": 0.0,\n        \"recall\": 0.0,\n        \"f1-score\": 0.0,\n        \"support\": 10.0\n      },\n      \"BIOMEDICO\": {\n        \"precision\": 0.25,\n        \"recall\": 1.0,\n        \"f1-score\": 0.4,\n        \"support\": 10.0\n      },\n      \"accuracy\": 0.25,\n      \"macro avg\": {\n        \"precision\": 0.0625,\n        \"recall\": 0.25,\n        \"f1-score\": 0.1,\n        \"support\": 40.0\n      },\n      \"weighted avg\": {\n        \"precision\": 0.0625,\n        \"recall\": 0.25,\n        \"f1-score\": 0.1,\n        \"support\": 40.0\n      }\n    },\n    \"labels\": [\n      \"SOPORTE\",\n      \"REDES\",\n      \"INFRAESTRUCTURA\",\n      \"BIOMEDICO\"\n    ],\n    \"confusion_matrix\": [\n      [\n        0,\n        0,\n        0,\n        10\n      ],\n      [\n        0,\n        0,\n        0,\n        10\n      ],\n      [\n        0,\n        0,\n        0,\n        10\n      ],\n      [\n        0,\n        0,\n        0,\n        10\n      ]\n    ],\n    \"confusion_matrix_normalized\": [\n      [\n        0.0,\n        0.0,\n        0.0,\n        1.0\n      ],\n      [\n        0.0,\n        0.0,\n        0.0,\n        1.0\n      ],\n      [\n        0.0,\n        0.0,\n        0.0,\n        1.0\n      ],\n      [\n        0.0,\n        0.0,\n        0.0,\n        1.0\n      ]\n    ]\n  },\n  \"cv_acceptance_passed\": true,\n  \"final_acceptance_passed\": true,\n  \"experimental_promotion_allowed\": true,\n  \"institutionally_validated\": false,\n  \"model_exported\": false,\n  \"status\": \"EXPERIMENTAL_CANDIDATE_ACCEPTED\"\n}\n","ml/experiments/experiment-v1.0.0/local-2026-10-07/final-test-started.json":"{\"opened_at_utc\": \"2026-10-08T03:43:36.603185+00:00\", \"selection_sha256\": \"64e14be457ebe135f05fdf94bf5c6acbec787f3a78d87a044ef20c8a91b440d0\"}","ml/experiments/experiment-v1.0.0/local-2026-10-07/result-manifest.json":"{\n  \"scope\": \"Real local stage 4.4 results; Colab validation pending\",\n  \"files\": {\n    \"comparison.csv\": {\n      \"sha256\": \"49d11985d7ad3b4d97ce9c558184fbf99773f4bbfc972b56ee756a90f659ccf8\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"comparison.json\": {\n      \"sha256\": \"a9877add46b308b57b2dcfbb8a6259b52f102db5a7387df04028dcfdcae1640f\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"comparison.pdf\": {\n      \"sha256\": \"99a0c1de8ab92b762f92b568e30a46dc174c6fd0a8c88365eb7dfd91a956cd50\",\n      \"hash_algorithm\": \"SHA-256\"\n    },\n    \"comparison.png\": {\n      \"sha256\": \"95f8b47990269c80a6d446aa1adc2aa636571c84484fcfa429b0b8301581650e\",\n      \"hash_algorithm\": \"SHA-256\"\n    },\n    \"confusion_matrix.csv\": {\n      \"sha256\": \"dfc870daea1cc6d26af6c13a2dc1ecbb28f61d7952f7bae8c4e6db5d081b8d47\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"confusion_matrix.pdf\": {\n      \"sha256\": \"589f7d5a5836efa9d63923cc803ce289bbcb7cea82a1673cef5863ebdded60f6\",\n      \"hash_algorithm\": \"SHA-256\"\n    },\n    \"confusion_matrix.png\": {\n      \"sha256\": \"cc255f76d5a5dc4bee316237a901218368d619262680f0a3a2320416428a2ae7\",\n      \"hash_algorithm\": \"SHA-256\"\n    },\n    \"confusion_matrix_normalized.csv\": {\n      \"sha256\": \"7195c799b7260120057a14d3b18dc98f03f7affe8f6ea2ad7b409b846bacf62a\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"confusion_matrix_normalized.pdf\": {\n      \"sha256\": \"7de17db3926a2e265fb5e6f51f1422a32922874a81c83a6ef7f18edab7a425f6\",\n      \"hash_algorithm\": \"SHA-256\"\n    },\n    \"confusion_matrix_normalized.png\": {\n      \"sha256\": \"07bcbd5e6611a883037462d47367986dd066af04f1b324b8d4356085d86eb9c8\",\n      \"hash_algorithm\": \"SHA-256\"\n    },\n    \"development-errors.csv\": {\n      \"sha256\": \"370efdb0073aea099744b968ec45ffd3a84692afbdcadc478dfb9a14a6dabd64\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"development-examples.csv\": {\n      \"sha256\": \"d8df49186e35c317477cd519d0f6edf81e737dd44298af8362ef0e863b0c84c1\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"development-oof.csv\": {\n      \"sha256\": \"da6d1c115a532c51e6cfa2681ebc03d92ba2c8704707945ab073e2f665e074e4\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"error-analysis.json\": {\n      \"sha256\": \"96540c55cfaaf83463805a8030bc1e1e670fad8c78400d09231e4ee17c0a3916\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"ERROR-ANALYSIS.md\": {\n      \"sha256\": \"cd7571c72fb70b9befee560f693530cf97215577d9f4e86826c0b8ae0c7a7de7\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"final-errors.csv\": {\n      \"sha256\": \"6f7266a05a2fedc223b7382bda8aebc64e9b9c3f124f388bfaac0ac27fa46c5b\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"final-evaluation.json\": {\n      \"sha256\": \"d051c1d48c736b4b7bec85dd859de48b530f6deb877f6cf0d2c042047ee27995\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"final-examples.csv\": {\n      \"sha256\": \"5f55da8d62086935f8ad6b662910ccb116e0bbff4efa1a11e2c166911819f2ee\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"final-metrics.json\": {\n      \"sha256\": \"1298513d33dce100cb5ae57b90f0d08387d726ecbce6013aead36f4a67642a63\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"final-test-started.json\": {\n      \"sha256\": \"d6607166618ca23f000e5bc930b0baae23a7bb79c1075a32dd8d8338b8019a88\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"REPORT.md\": {\n      \"sha256\": \"93d6eed0b45c2e6adf2a6587592eb7aa979aa2e4a92bb87c9fdeac23056ac95c\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    },\n    \"selection.json\": {\n      \"sha256\": \"a0b4970ab4941103c87a3ffe052f698f0f14c062c031d63016c77bc99505bea2\",\n      \"hash_algorithm\": \"SHA-256-UTF8-LF\"\n    }\n  }\n}\n","ml/modeling/runtime.py":"\"\"\"Trusted local model loading only. No HTTP, database, pickle or joblib.load.\"\"\"\nfrom hashlib import sha256\nfrom importlib import metadata\nfrom io import BytesIO\nimport json\nfrom pathlib import Path, PurePosixPath\nimport sys\nimport zipfile\n\nMODEL_VERSION = \"ticket-category-model-v1.0.0\"\nLABELS = [\"SOPORTE\", \"REDES\", \"INFRAESTRUCTURA\", \"BIOMEDICO\"]\nRUNTIME_PINS = {\"numpy\": \"2.2.6\", \"scipy\": \"1.15.3\", \"scikit-learn\": \"1.6.1\",\n                \"joblib\": \"1.5.1\", \"threadpoolctl\": \"3.6.0\", \"packaging\": \"25.0\",\n                \"skops\": \"0.16.0\", \"prettytable\": \"3.18.0\", \"wcwidth\": \"0.9.2\"}\n# Reviewed after a real roundtrip; never trust types merely because a file requests them.\nREVIEWED_EXTRA_TYPES = frozenset()\nFILES = {\"model.skops\", \"metadata.json\", \"verification.json\"}\nMAX_MODEL_BYTES = 8 * 1024 * 1024\n\n\ndef require(condition, code):\n    if not condition:\n        raise ValueError(code)\n\n\ndef text_hash(payload):\n    return sha256(payload.decode(\"utf-8\").replace(\"\\r\\n\", \"\\n\").encode(\"utf-8\")).hexdigest()\n\n\ndef verify_runtime():\n    require(sys.version_info[:2] == (3, 12), \"MODEL_REQUIRES_PYTHON_3_12\")\n    try:\n        installed = {name: metadata.version(name) for name in RUNTIME_PINS}\n    except metadata.PackageNotFoundError:\n        raise ValueError(\"MODEL_DEPENDENCY_MISSING: run ml:model:setup\") from None\n    require(installed == RUNTIME_PINS, \"MODEL_DEPENDENCY_VERSION_MISMATCH\")\n    return installed\n\n\ndef inspect_archive(payload):\n    require(0 < len(payload) <= MAX_MODEL_BYTES, \"MODEL_SIZE_INVALID\")\n    try:\n        with zipfile.ZipFile(BytesIO(payload)) as archive:\n            entries = archive.infolist()\n            require(0 < len(entries) <= 100, \"MODEL_ARCHIVE_ENTRY_LIMIT\")\n            require(len({e.filename for e in entries}) == len(entries), \"MODEL_ARCHIVE_DUPLICATE\")\n            require(all(not e.flag_bits & 1 and e.file_size <= 16 * 1024 * 1024 for e in entries)\n                    and sum(e.file_size for e in entries) <= 16 * 1024 * 1024, \"MODEL_ARCHIVE_EXPANSION_LIMIT\")\n            require(all(e.orig_filename == e.filename and not e.filename.startswith((\"/\", \"\\\\\")) and \"\\\\\" not in e.filename and\n                        \":\" not in e.filename and \"..\" not in PurePosixPath(e.filename).parts for e in entries), \"MODEL_ARCHIVE_PATH_INVALID\")\n            require(\"schema.json\" in {e.filename for e in entries}, \"MODEL_ARCHIVE_SCHEMA_MISSING\")\n    except zipfile.BadZipFile:\n        raise ValueError(\"MODEL_ARCHIVE_INVALID\") from None\n\n\ndef deserialize(payload):\n    verify_runtime()\n    inspect_archive(payload)\n    import skops.io as sio\n    unexpected = set(sio.get_untrusted_types(data=payload)) - REVIEWED_EXTRA_TYPES\n    require(not unexpected, \"MODEL_UNREVIEWED_TYPES\")\n    estimator = sio.loads(payload, trusted=sorted(REVIEWED_EXTRA_TYPES))\n    from sklearn.pipeline import Pipeline, FeatureUnion\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.svm import LinearSVC\n    require(type(estimator) is Pipeline and list(estimator.named_steps) == [\"features\", \"classifier\"], \"MODEL_PIPELINE_INVALID\")\n    features, classifier = estimator.named_steps[\"features\"], estimator.named_steps[\"classifier\"]\n    require(type(features) is FeatureUnion and [name for name, _ in features.transformer_list] == [\"word\", \"char\"], \"MODEL_FEATURES_INVALID\")\n    require(all(type(v) is TfidfVectorizer for _, v in features.transformer_list) and\n            type(classifier) is LinearSVC and classifier.C == 1.0, \"MODEL_ESTIMATOR_INVALID\")\n    require(classifier.classes_.tolist() == sorted(LABELS), \"MODEL_CLASSES_INVALID\")\n    import numpy as np\n    require(np.isfinite(classifier.coef_).all() and np.isfinite(classifier.intercept_).all(), \"MODEL_WEIGHTS_INVALID\")\n    return estimator\n\n\ndef load_release(directory, approval=None):\n    \"\"\"approval must come from reviewed application code, NEVER from an uploaded bundle.\"\"\"\n    if approval is None:\n        approval = json.loads(Path(__file__).with_name(\"approved-release.json\").read_text(encoding=\"utf-8\"))\n    require(approval.get(\"model_version\") == MODEL_VERSION and set(approval.get(\"files\", {})) == FILES, \"MODEL_APPROVAL_INVALID\")\n    root = Path(directory).resolve()\n    payloads = {}\n    for name in sorted(FILES):\n        path = root / name\n        limit = MAX_MODEL_BYTES if name == \"model.skops\" else 512 * 1024\n        require(path.is_file() and not path.is_symlink() and path.stat().st_size <= limit, \"MODEL_FILE_INVALID\")\n        payload = path.read_bytes()\n        actual = sha256(payload).hexdigest() if name == \"model.skops\" else text_hash(payload)\n        require(actual == approval[\"files\"][name], \"MODEL_CHECKSUM_MISMATCH\")\n        payloads[name] = payload\n    info = json.loads(payloads[\"metadata.json\"])\n    require(info[\"model_version\"] == MODEL_VERSION and info[\"runtime\"][\"packages\"] == RUNTIME_PINS and\n            info[\"runtime\"][\"python_minor\"] == \"3.12\" and info[\"categories\"] == LABELS, \"MODEL_METADATA_INVALID\")\n    return CategoryModel(deserialize(payloads[\"model.skops\"]), info)\n\n\nclass CategoryModel:\n    def __init__(self, estimator, info):\n        self.estimator, self.info = estimator, info\n\n    def predict(self, titulo, descripcion):\n        require(isinstance(titulo, str) and isinstance(descripcion, str), \"MODEL_TEXT_TYPE_INVALID\")\n        # Bound before trimming to prevent unbounded whitespace; same domain limits after trim.\n        require(len(titulo) <= 200 and len(descripcion) <= 5000 and\n                5 <= len(titulo.strip()) and 10 <= len(descripcion.strip()), \"MODEL_TEXT_LENGTH_INVALID\")\n        text = titulo.strip() + \"\\n\" + descripcion.strip()\n        category = self.estimator.predict([text])[0]\n        scores = self.estimator.decision_function([text])[0]\n        return {\"categoria_sugerida\": str(category), \"score\": float(max(scores)),\n                \"score_type\": \"raw_decision_margin_not_probability\", \"probability\": None,\n                \"operational_threshold\": None, \"model_version\": MODEL_VERSION}\n","ml/modeling/bootstrap.py":"\"\"\"Install stage 4.5 into its own venv; preserve all stage 4.3/4.4 pins.\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\n\n\ndef pins(text):\n    result = {}\n    for line in text.splitlines():\n        line = line.strip()\n        if not line or line.startswith(\"#\"):\n            continue\n        match = re.fullmatch(r\"([a-zA-Z0-9-]+)==([0-9]+(?:\\.[0-9]+)+(?:\\.post[0-9]+)?)\", line)\n        if match is None or match[1] in result:\n            raise ValueError(\"INVALID_MODEL_DEPENDENCY_PIN\")\n        result[match[1]] = match[2]\n    if len(result) != 21:\n        raise ValueError(\"INCOMPLETE_MODEL_DEPENDENCY_PINS\")\n    return result\n\n\ndef prepare(workspace, requirements, install=True):\n    if sys.version_info[:2] != (3, 12):\n        raise ValueError(\"USE_PYTHON_3_12: install Python 3.12 or configure ML_PYTHON\")\n    pins(requirements)\n    workspace = Path(workspace).resolve()\n    workspace.mkdir(parents=True, exist_ok=True)\n    python = workspace / \".venv\" / (\"Scripts/python.exe\" if sys.platform == \"win32\" else \"bin/python\")\n    if not python.exists():\n        subprocess.run([sys.executable, \"-m\", \"venv\", \"--without-pip\", str(python.parents[1])], check=True, timeout=120)\n    requirement_file = workspace / \"requirements-stage-4.5.txt\"\n    requirement_file.write_text(requirements, encoding=\"utf-8\")\n    if install:\n        subprocess.run([sys.executable, \"-m\", \"pip\", \"--isolated\", \"--python\", str(python), \"install\",\n                        \"--disable-pip-version-check\", \"--only-binary=:all:\", \"--index-url\", \"https://pypi.org/simple\",\n                        \"pip==25.1.1\", \"-r\", str(requirement_file)], check=True, timeout=600)\n    verify(python, requirements)\n    return python\n\n\ndef verify(python, requirements):\n    expected = pins(requirements)\n    code = \"import json,importlib.metadata as m,sys;assert sys.version_info[:2]==(3,12), 'USE_PYTHON_3_12';assert m.version('pip')=='25.1.1', 'INSTALLER_VERSION_MISMATCH';print(json.dumps({n:m.version(n) for n in json.loads(sys.argv[1])}))\"\n    run = subprocess.run([str(python), \"-c\", code, json.dumps(list(expected))], check=True, capture_output=True, text=True, timeout=60)\n    if json.loads(run.stdout) != expected:\n        raise ValueError(\"MODEL_ENVIRONMENT_VERSION_MISMATCH\")\n    subprocess.run([str(python), \"-m\", \"pip\", \"check\"], check=True, timeout=60)\n    return expected\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--workspace\", type=Path, required=True)\n    parser.add_argument(\"--requirements\", type=Path, default=Path(__file__).with_name(\"requirements.txt\"))\n    parser.add_argument(\"--no-install\", action=\"store_true\")\n    args = parser.parse_args()\n    python = prepare(args.workspace, args.requirements.read_text(encoding=\"utf-8\"), not args.no_install)\n    print(f\"MODEL_ENVIRONMENT_READY: {python}\")\n","ml/modeling/release.py":"\"\"\"Stage 4.5: rebuild the accepted 4.4 candidate; never select or evaluate again.\"\"\"\nimport argparse\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom hashlib import sha256\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport platform\nimport sys\nimport zipfile\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport bootstrap\nimport runtime\n\nEXPERIMENT = \"ml/experiments/experiment-v1.0.0\"\nRESULTS = EXPERIMENT + \"/local-2026-10-07\"\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\"utf-8\"))\n\n\ndef write(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + \"\\n\", encoding=\"utf-8\")\n\n\ndef locked_inputs(root):\n    \"\"\"The lock is reviewed repository code, not a manifest received with an artifact.\"\"\"\n    root = Path(root)\n    lock = read(Path(__file__).with_name(\"source-lock.json\"))\n    for name, expected in lock[\"files\"].items():\n        runtime.require(runtime.text_hash((root / name).read_bytes()) == expected, \"MODEL_SOURCE_HASH_MISMATCH\")\n    spec = importlib.util.spec_from_file_location(\"stage44_model_source\", root / \"ml/experimentation/experiment.py\")\n    exp = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(exp)\n    records, dataset = exp.prep.verify_dataset(exp.prep.load_local_snapshot(root))\n    protocol, splits = read(root / EXPERIMENT / \"protocol.json\"), read(root / EXPERIMENT / \"splits.json\")\n    summary = exp.validate_splits(records, protocol, splits)\n    selection, comparison = read(root / RESULTS / \"selection.json\"), read(root / RESULTS / \"comparison.json\")\n    final = read(root / RESULTS / \"final-metrics.json\")\n    decision = selection[\"decision\"]\n    runtime.require(final[\"status\"] == \"EXPERIMENTAL_CANDIDATE_ACCEPTED\" and\n                    final[\"experimental_promotion_allowed\"] is True and decision[\"cv_acceptance_passed\"] is True and\n                    final[\"candidate_id\"] == decision[\"candidate_id\"] == \"linear-svc-c1\" and\n                    selection[\"final_test_used_for_selection\"] is False, \"MODEL_CANDIDATE_NOT_APPROVED\")\n    runtime.require(final[\"selection_sha256_canonical_json\"] == selection[\"selection_sha256_canonical_json\"] and\n                    exp.digest(comparison) == selection[\"comparison_sha256_canonical_json\"], \"MODEL_SELECTION_MISMATCH\")\n    return exp, records, dataset, protocol, splits, summary, selection, comparison, final, lock\n\n\ndef training_rows(records, splits):\n    # The reserved test is excluded before extracting text, labels or fitting vocabulary.\n    by_id = {r[\"record_id\"]: r for r in records if r[\"record_id\"] in set(splits[\"development_ids\"])}\n    runtime.require(len(by_id) == 160 and not set(by_id) & set(splits[\"final_test_ids\"]), \"MODEL_TRAINING_SPLIT_INVALID\")\n    return [by_id[i] for i in splits[\"development_ids\"]]\n\n\ndef build(root, output):\n    root, output = Path(root).resolve(), Path(output).resolve()\n    runtime.require(not output.exists(), \"MODEL_OUTPUT_ALREADY_EXISTS: use a new directory; releases are immutable\")\n    runtime.verify_runtime()\n    requirement_text = Path(__file__).with_name(\"requirements.txt\").read_text(encoding=\"utf-8\")\n    environment = bootstrap.verify(sys.executable, requirement_text)\n    exp, records, dataset, protocol, splits, summary, selection, comparison, final, lock = locked_inputs(root)\n    inherited_pins = exp.prep.parse_requirements((root / \"ml/colab/requirements.txt\").read_text(encoding=\"utf-8\"))\n    runtime.require(all(environment.get(k) == v for k, v in inherited_pins.items()), \"MODEL_CHANGED_EXPERIMENT_DEPENDENCY\")\n    development = training_rows(records, splits)\n    estimator = exp.make_estimator(selection[\"decision\"][\"candidate_spec\"], protocol)\n    texts = [exp.text_input(r) for r in development]\n    estimator.fit(texts, [r[\"categoria\"] for r in development])\n    import skops.io as sio\n    import numpy as np\n    payload = sio.dumps(estimator, compression=zipfile.ZIP_DEFLATED, compresslevel=9)\n    restored = runtime.deserialize(payload)\n    original_scores, loaded_scores = estimator.decision_function(texts), restored.decision_function(texts)\n    predictions_identical = np.array_equal(estimator.predict(texts), restored.predict(texts))\n    max_delta = float(np.max(np.abs(original_scores - loaded_scores)))\n    runtime.require(predictions_identical and np.allclose(original_scores, loaded_scores, rtol=1e-12, atol=1e-12), \"MODEL_ROUNDTRIP_MISMATCH\")\n    chosen = next(row for row in comparison[\"candidates\"] if row[\"id\"] == final[\"candidate_id\"])\n    created = datetime.now(timezone.utc).isoformat()\n    info = {\n        \"schema_version\": 1, \"model_version\": runtime.MODEL_VERSION, \"created_at_utc\": created,\n        \"status\": \"EXPERIMENTAL_ARTIFACT_READY\", \"institutionally_validated\": False,\n        \"synthetic_academic_only\": True, \"categories\": runtime.LABELS,\n        \"estimator_class_order\": estimator.classes_.tolist(),\n        \"input\": {\"fields\": [\"titulo\", \"descripcion\"], \"text_join\": protocol[\"text_join\"], \"language\": \"es\",\n                  \"titulo_length\": [5, 200], \"descripcion_length\": [10, 5000]},\n        \"output\": {\"score_type\": \"raw_decision_margin_not_probability\", \"probability\": None,\n                   \"operational_threshold\": None, \"human_decision_required\": True},\n        \"algorithm\": selection[\"decision\"][\"candidate_spec\"], \"vectorizer\": protocol[\"vectorizer\"], \"seed\": protocol[\"seed\"],\n        \"dataset\": {k: v for k, v in dataset.items() if k not in {\"training_started\", \"splits_created\"}},\n        \"training\": {\"source\": \"stage_4_4_development_partition_only\", \"record_ids\": splits[\"development_ids\"],\n                     \"records\": len(development), \"distribution\": dict(Counter(r[\"categoria\"] for r in development)),\n                     \"groups\": summary[\"development\"][\"groups\"], \"reserved_test_records\": 40,\n                     \"final_test_used_for_fit\": False, \"training_ids_sha256_canonical_json\": exp.digest(splits[\"development_ids\"])},\n        \"evaluation\": {\"source\": RESULTS, \"experiment_version\": exp.VERSION, \"new_evaluation_in_stage_4_5\": False,\n                       \"mean_cv_macro_f1\": chosen[\"mean_macro_f1\"], \"std_cv_macro_f1\": chosen[\"std_macro_f1\"],\n                       \"development_oof_metrics\": chosen[\"oof_metrics\"], \"final_test_metrics\": final[\"metrics\"],\n                       \"final_test_records\": 40, \"final_test_groups\": 8,\n                       \"selection_sha256_canonical_json\": selection[\"selection_sha256_canonical_json\"]},\n        \"provenance\": {\"experiment_commit\": lock[\"experiment_commit\"], \"experiment_inputs\": lock[\"files\"],\n                       \"modeling_sources_sha256_utf8_lf\": {n: runtime.text_hash(Path(__file__).with_name(n).read_bytes())\n                                                           for n in [\"release.py\", \"runtime.py\", \"bootstrap.py\", \"source-lock.json\", \"requirements.txt\"]}},\n        \"build_environment\": {\"python\": platform.python_version(), \"platform\": platform.system(), \"packages\": environment},\n        \"runtime\": {\"python_minor\": \"3.12\", \"packages\": runtime.RUNTIME_PINS},\n        \"artifact\": {\"file\": \"model.skops\", \"format\": \"skops\", \"sha256\": sha256(payload).hexdigest(), \"bytes\": len(payload)}\n    }\n    verification = {\"model_version\": runtime.MODEL_VERSION, \"check\": \"serialization_roundtrip_on_development_only\",\n                    \"records\": 160, \"predictions_identical\": bool(predictions_identical), \"max_absolute_margin_delta\": max_delta,\n                    \"margin_tolerance\": {\"rtol\": 1e-12, \"atol\": 1e-12}, \"unreviewed_types\": [],\n                    \"final_test_reopened\": False, \"new_metrics_calculated\": False,\n                    \"operational_database_access\": False, \"status\": \"MODEL_ROUNDTRIP_VERIFIED\"}\n    output.mkdir(parents=True)\n    (output / \"model.skops\").write_bytes(payload)\n    write(output / \"metadata.json\", info)\n    write(output / \"verification.json\", verification)\n    approval = {\"schema_version\": 1, \"model_version\": runtime.MODEL_VERSION,\n                \"files\": {n: sha256(payload).hexdigest() if n == \"model.skops\" else runtime.text_hash((output / n).read_bytes())\n                          for n in sorted(runtime.FILES)}}\n    # Receipt is diagnostic; it is NEVER automatically trusted by the runtime loader.\n    write(output / \"build-receipt.json\", approval)\n    model = runtime.load_release(output, approval=approval)\n    probe = model.predict(\"Impresora no responde\", \"La impresora local no imprime desde el equipo de prueba.\")\n    print(json.dumps({\"status\": \"MODEL_V1_BUILT\", \"model_version\": runtime.MODEL_VERSION,\n                      \"training_records\": 160, \"reserved_test_records\": 40, \"bytes\": len(payload),\n                      \"sha256\": approval[\"files\"][\"model.skops\"], \"verification\": verification, \"smoke_prediction\": probe}, ensure_ascii=False, indent=2))\n    return approval\n\n\ndef check(directory):\n    model = runtime.load_release(directory)\n    proof = read(Path(directory) / \"verification.json\")\n    runtime.require(proof[\"status\"] == \"MODEL_ROUNDTRIP_VERIFIED\" and proof[\"predictions_identical\"] is True, \"MODEL_VERIFICATION_INVALID\")\n    probe = model.predict(\"Equipo no inicia\", \"El equipo de demostración no termina de iniciar el sistema operativo.\")\n    print(json.dumps({\"status\": \"MODEL_V1_VERIFIED\", \"model_version\": model.info[\"model_version\"],\n                      \"training_records\": model.info[\"training\"][\"records\"], \"probe\": probe}, ensure_ascii=False, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--phase\", choices=[\"build\", \"check\"], required=True)\n    parser.add_argument(\"--root\", type=Path, default=Path(__file__).resolve().parents[2])\n    parser.add_argument(\"--output\", type=Path)\n    parser.add_argument(\"--release\", type=Path)\n    args = parser.parse_args()\n    if args.phase == \"build\":\n        runtime.require(args.output is not None, \"MODEL_OUTPUT_REQUIRED\")\n        build(args.root, args.output)\n    else:\n        check(args.release or args.root / \"ml/models/category-v1.0.0\")\n\n\nif __name__ == \"__main__\":\n    try:\n        main()\n    except Exception as error:\n        print(f\"MODEL_FAILED: {type(error).__name__}: {error}\", file=sys.stderr)\n        sys.exit(1)\n","ml/modeling/source-lock.json":"{\n  \"experiment_commit\": \"1d4e5508bbfb41d219fa073436d2a9a620118730\",\n  \"hash_algorithm\": \"SHA-256-UTF8-LF\",\n  \"files\": {\n    \"ml/colab/preparation.py\": \"e2ac1f6e1f2eb0da013e253869899ab4800c0e007c3a959d4c459a9dcaf06c98\",\n    \"ml/colab/requirements.txt\": \"38e73e22486a6b4346b321c94b95223ad53a53954e293e06af91c8fd69c4c86b\",\n    \"ml/experimentation/experiment.py\": \"236d6c62a45b014eb74f5cf6b56ba305d948554a5aa4af87c75a0ae3eb9e37d0\",\n    \"ml/experiments/experiment-v1.0.0/protocol.json\": \"57cba9fda793ffdd0ea796349209c911586c99ed5862f20756ed03b86f5de56f\",\n    \"ml/experiments/experiment-v1.0.0/splits.json\": \"041d520e633e213dd51566bad01c103d13cc7e7f2463d0553b2581bb0c8fca81\",\n    \"ml/experiments/experiment-v1.0.0/local-2026-10-07/comparison.json\": \"a9877add46b308b57b2dcfbb8a6259b52f102db5a7387df04028dcfdcae1640f\",\n    \"ml/experiments/experiment-v1.0.0/local-2026-10-07/selection.json\": \"a0b4970ab4941103c87a3ffe052f698f0f14c062c031d63016c77bc99505bea2\",\n    \"ml/experiments/experiment-v1.0.0/local-2026-10-07/final-metrics.json\": \"1298513d33dce100cb5ae57b90f0d08387d726ecbce6013aead36f4a67642a63\",\n    \"ml/experiments/experiment-v1.0.0/local-2026-10-07/final-test-started.json\": \"d6607166618ca23f000e5bc930b0baae23a7bb79c1075a32dd8d8338b8019a88\",\n    \"ml/experiments/experiment-v1.0.0/local-2026-10-07/result-manifest.json\": \"21005f2c20f4e96148a93b1dbccbb5b2d0ef85b2425f8c9122fa9bb3407ecdae\"\n  }\n}\n","ml/modeling/requirements.txt":"# Model build environment for stage 4.5; Python 3.12; CPU only.\n# Preserve the 18 experiment pins; add only persistence dependencies.\nnumpy==2.2.6\nscipy==1.15.3\npandas==2.2.3\nscikit-learn==1.6.1\nmatplotlib==3.10.3\njoblib==1.5.1\nthreadpoolctl==3.6.0\ncontourpy==1.3.2\ncycler==0.12.1\nfonttools==4.58.0\nkiwisolver==1.4.8\npackaging==25.0\npillow==11.2.1\npyparsing==3.2.3\npython-dateutil==2.9.0.post0\npytz==2025.2\nsix==1.17.0\ntzdata==2025.2\n\nskops==0.16.0\nprettytable==3.18.0\nwcwidth==0.9.2\n"}
EMBEDDED_HASHES = {"ml/colab/preparation.py":"e2ac1f6e1f2eb0da013e253869899ab4800c0e007c3a959d4c459a9dcaf06c98","ml/colab/requirements.txt":"38e73e22486a6b4346b321c94b95223ad53a53954e293e06af91c8fd69c4c86b","ml/experimentation/experiment.py":"236d6c62a45b014eb74f5cf6b56ba305d948554a5aa4af87c75a0ae3eb9e37d0","ml/experiments/experiment-v1.0.0/protocol.json":"57cba9fda793ffdd0ea796349209c911586c99ed5862f20756ed03b86f5de56f","ml/experiments/experiment-v1.0.0/splits.json":"041d520e633e213dd51566bad01c103d13cc7e7f2463d0553b2581bb0c8fca81","ml/experiments/experiment-v1.0.0/local-2026-10-07/comparison.json":"a9877add46b308b57b2dcfbb8a6259b52f102db5a7387df04028dcfdcae1640f","ml/experiments/experiment-v1.0.0/local-2026-10-07/selection.json":"a0b4970ab4941103c87a3ffe052f698f0f14c062c031d63016c77bc99505bea2","ml/experiments/experiment-v1.0.0/local-2026-10-07/final-metrics.json":"1298513d33dce100cb5ae57b90f0d08387d726ecbce6013aead36f4a67642a63","ml/experiments/experiment-v1.0.0/local-2026-10-07/final-test-started.json":"d6607166618ca23f000e5bc930b0baae23a7bb79c1075a32dd8d8338b8019a88","ml/experiments/experiment-v1.0.0/local-2026-10-07/result-manifest.json":"21005f2c20f4e96148a93b1dbccbb5b2d0ef85b2425f8c9122fa9bb3407ecdae","ml/modeling/runtime.py":"a850ec04ae3074fbd15855b6df75a9c0568009f1f3a66faf08b1c23e7b18983e","ml/modeling/bootstrap.py":"7335c95d6d379380fc15b12a0c1c09c4dd5ad67fb087aef8a11ccbc6630554db","ml/modeling/release.py":"c789c7f67520ffc4788f7af4edc4682fea623d39875c427f778e886ff963c78a","ml/modeling/source-lock.json":"5a8aa2f25838ac663089e233f9ae5f0d6f7f35f7eb59e2a4db793a4bc9ff59a8","ml/modeling/requirements.txt":"e2d040c98781c3e26887e74d193fde448c90173bc8ed1a0937275fed83a34308"}
input_directory = WORKSPACE / "inputs"
for name, content in EMBEDDED_FILES.items():
    require(canonical_hash(content.encode()) == EMBEDDED_HASHES[name], "MODEL_NOTEBOOK_SOURCE_MISMATCH")
    destination = input_directory / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")
for name, payload in payloads.items():
    destination = input_directory / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(payload)
modeling_directory = input_directory / "ml/modeling"
import importlib.util
spec = importlib.util.spec_from_file_location("model_bootstrap", modeling_directory / "bootstrap.py")
model_bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_bootstrap)


## Entorno aislado

Python 3.12, 21 versiones fijadas, pip check. Los paquetes globales de Colab no se alteran.


In [ ]:
model_python = model_bootstrap.prepare(WORKSPACE, EMBEDDED_FILES["ml/modeling/requirements.txt"], install=INSTALL_DEPENDENCIES)
print("MODEL_ENVIRONMENT_READY", model_python)


## Construir y comprobar el artefacto

Ajuste de LinearSVC C=1 sobre 160 ejemplos de desarrollo, roundtrip sobre esos mismos textos. No se calculan nuevas métricas ni se usa el holdout. El margen puede ser negativo y no es probabilidad. Para repetir desde cero, elimina el entorno de ejecución de Colab y empieza en una sesión nueva.


In [ ]:
model_directory = WORKSPACE / "rebuilt-model-v1"
subprocess.run([str(model_python), "-B", str(modeling_directory / "release.py"), "--phase", "build", "--root", str(input_directory), "--output", str(model_directory)], check=True)
model_metadata = json.loads((model_directory / "metadata.json").read_text())
verification = json.loads((model_directory / "verification.json").read_text())
require(verification["status"] == "MODEL_ROUNDTRIP_VERIFIED", "MODEL_NOT_VERIFIED")
print(json.dumps({"status": "MODEL_V1_REPRODUCED", "model_version": model_metadata["model_version"], "training_records": model_metadata["training"]["records"], "artifact": model_metadata["artifact"], "verification": verification}, ensure_ascii=False, indent=2))


## Descargar evidencia opcional

Activa DOWNLOAD_MODEL y ejecuta solo esta celda. El ZIP contiene un artefacto reconstruido y su recibo diagnóstico; no es una aprobación para cargar archivos arbitrarios ni reemplaza automáticamente Model V1.


In [ ]:
DOWNLOAD_MODEL = False
if DOWNLOAD_MODEL:
    import zipfile
    archive_path = WORKSPACE / "stage-4.5-rebuilt-model-v1.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ["model.skops", "metadata.json", "verification.json", "build-receipt.json"]:
            archive.write(model_directory / name, name)
    from google.colab import files
    files.download(str(archive_path))
else:
    print("Descarga desactivada; artefacto verificado y conservado en", model_directory)
